In [2]:
import pandas as pd
import numpy as np

# =========================================================
# 0) 基本設定
# =========================================================

input_path = "C:/Users/User/Desktop/LC analysis/result/normal_highrisk_five_features_results.xlsx"
# sheet_name = "merged_row_level"   # 若要用鄉鎮平均資料，可改成 "merged_by_town"

output_path = "C:/Users/User/Desktop/LC analysis/result/回歸用資料.xlsx"
stats_output_path = "變數轉換參數表.xlsx"

# 若你要用 2015-2017 的平均與標準差去轉 2018，設成 [2015, 2016, 2017]
# 若要用全資料計算平均與標準差，設成 None
FIT_ON_YEARS = None
# FIT_ON_YEARS = [2015, 2016, 2017]

EPS = 1e-6   # 避免 logit(0) 或 logit(1) 爆掉


# =========================================================
# 1) 讀取資料
# =========================================================
df = pd.read_excel(input_path)

# df = pd.read_excel(input_path, sheet_name=sheet_name)

# 年份處理：支援 2015、"2015"、datetime
if "年份" in df.columns:
    year_num = pd.to_datetime(df["年份"], errors="coerce").dt.year
    if year_num.isna().all():
        year_num = pd.to_numeric(df["年份"], errors="coerce")
else:
    year_num = pd.Series(np.nan, index=df.index)

if FIT_ON_YEARS is None:
    fit_mask = pd.Series(True, index=df.index)
else:
    fit_mask = year_num.isin(FIT_ON_YEARS)


# =========================================================
# 2) 指定變數類型
# =========================================================

# 這些變數直接當連續型，做 z-score
continuous_vars = [
    "該鄉鎮總人口數 (人數)",
    "小於18歲人口數 (人數)",
    "Income_regular (百萬元)",
    "平均年齡（歲）",
    "平均BMI  (kg/m2)",
    "平均健保投保金額",

    "Lead_water",
    "PM2.5 (ug/m3)",
    "每百萬人病床數 (床/萬人)",
    "平均日常體能狀態（分）",
    "加權 CCI",
    "平均診斷到index date時間(日)",
]

# Lead_soil：不知道單位，所以用資料最大值當上界，轉成比例後 logit
max_to_ratio_vars = [
    "Lead_soil"
]

# 這些原本就是比例型變數，直接 logit 轉連續
# 這裡先排除死亡，因為死亡通常是 response / y，不一定要當 X 轉換
proportion_vars = [
    "性別(男)",

    "癌症期別1",
    "癌症期別2",

    "仍吸菸",
    "仍喝酒",

    "癌症組織類型1",
    "癌症組織類型2",
    "癌症組織類型3",
    "癌症組織類型4",

    "OP指標日期前",
    "RT指標日期後",

    "併用藥物1",
    "併用藥物2",
    "併用藥物3",
    "併用藥物4",
    "併用藥物5",
    "併用藥物6",
    "併用藥物7",
    "併用藥物8",
    "併用藥物9",
    "併用藥物10",
    "併用藥物11",

    "醫學中心",
]

# 如果你也想把死亡率重新做 logit，可打開這行
# proportion_vars = proportion_vars + ["死亡"]


# =========================================================
# 3) 小工具函數
# =========================================================

def keep_existing_columns(df, cols):
    """只保留資料中真的存在的欄位，避免不同 sheet 欄位不完全相同時報錯。"""
    return [c for c in cols if c in df.columns]


def to_numeric_series(s):
    """把欄位轉成 numeric。無法轉的值變成 NaN。"""
    return pd.to_numeric(s, errors="coerce")


def z_score_by_fit(x, fit_mask):
    """
    用 fit_mask 指定的資料計算 mean / sd，
    再套用到全部資料。
    """
    x = to_numeric_series(x)
    mu = x.loc[fit_mask].mean(skipna=True)
    sd = x.loc[fit_mask].std(skipna=True)

    if pd.isna(sd) or sd == 0:
        z = pd.Series(np.nan, index=x.index)
    else:
        z = (x - mu) / sd

    return z, mu, sd


def safe_logit(p, eps=1e-6):
    """
    logit 轉換：
    p 會先被壓在 [eps, 1-eps]，避免 0 和 1 造成無限大。
    """
    p = to_numeric_series(p)
    p_clip = p.clip(lower=eps, upper=1 - eps)
    return np.log(p_clip / (1 - p_clip))


def auto_fix_percent_to_ratio(x):
    """
    若比例變數不小心是 0-100 的百分比，會自動除以 100。
    若本來就在 0-1，則維持原狀。
    """
    x = to_numeric_series(x)
    xmax = x.max(skipna=True)

    if pd.notna(xmax) and xmax > 1 and xmax <= 100:
        return x / 100
    else:
        return x


# =========================================================
# 4) 開始轉換
# =========================================================

transform_stats = []

# ---------- A. 連續變數：直接 z-score ----------
for col in keep_existing_columns(df, continuous_vars):
    z, mu, sd = z_score_by_fit(df[col], fit_mask)

    new_col = f"{col}_z"
    df[new_col] = z

    transform_stats.append({
        "variable": col,
        "new_variable": new_col,
        "type": "continuous_z",
        "mean_used": mu,
        "sd_used": sd,
        "max_used": np.nan,
        "note": "連續變數，直接 z-score 標準化"
    })


# ---------- B. Lead_soil：最大值轉比例，再 logit，再 z-score ----------
for col in keep_existing_columns(df, max_to_ratio_vars):
    x = to_numeric_series(df[col])

    max_value = x.loc[fit_mask].max(skipna=True)

    if pd.isna(max_value) or max_value == 0:
        ratio = pd.Series(np.nan, index=df.index)
    else:
        ratio = x / max_value

    ratio_col = f"{col}_ratio_by_max"
    logit_col = f"{col}_ratio_logit"
    logit_z_col = f"{col}_ratio_logit_z"

    df[ratio_col] = ratio
    df[logit_col] = safe_logit(ratio, eps=EPS)

    z, mu, sd = z_score_by_fit(df[logit_col], fit_mask)
    df[logit_z_col] = z

    transform_stats.append({
        "variable": col,
        "new_variable": ratio_col,
        "type": "max_to_ratio",
        "mean_used": np.nan,
        "sd_used": np.nan,
        "max_used": max_value,
        "note": "用該變數在 fit 資料中的最大值作為上界，轉成比例"
    })

    transform_stats.append({
        "variable": col,
        "new_variable": logit_col,
        "type": "ratio_logit",
        "mean_used": np.nan,
        "sd_used": np.nan,
        "max_used": max_value,
        "note": "Lead_soil 先轉比例，再做 logit"
    })

    transform_stats.append({
        "variable": col,
        "new_variable": logit_z_col,
        "type": "ratio_logit_z",
        "mean_used": mu,
        "sd_used": sd,
        "max_used": max_value,
        "note": "Lead_soil logit 後再 z-score，可作為模型解釋用變數"
    })


# ---------- C. 原本比例變數：logit，再 z-score ----------
for col in keep_existing_columns(df, proportion_vars):
    p = auto_fix_percent_to_ratio(df[col])

    logit_col = f"{col}_logit"
    logit_z_col = f"{col}_logit_z"

    df[logit_col] = safe_logit(p, eps=EPS)

    z, mu, sd = z_score_by_fit(df[logit_col], fit_mask)
    df[logit_z_col] = z

    transform_stats.append({
        "variable": col,
        "new_variable": logit_col,
        "type": "proportion_logit",
        "mean_used": np.nan,
        "sd_used": np.nan,
        "max_used": np.nan,
        "note": "比例變數，先確認是否為 0-100 百分比，再轉 0-1 後 logit"
    })

    transform_stats.append({
        "variable": col,
        "new_variable": logit_z_col,
        "type": "proportion_logit_z",
        "mean_used": mu,
        "sd_used": sd,
        "max_used": np.nan,
        "note": "比例變數 logit 後再 z-score，可作為模型解釋用變數"
    })


# =========================================================
# 5) 建立建模用欄位清單
# =========================================================

continuous_z_cols = [
    f"{c}_z" for c in keep_existing_columns(df, continuous_vars)
]

lead_soil_model_cols = [
    f"{c}_ratio_logit_z" for c in keep_existing_columns(df, max_to_ratio_vars)
]

proportion_logit_z_cols = [
    f"{c}_logit_z" for c in keep_existing_columns(df, proportion_vars)
]

model_x_cols = continuous_z_cols + lead_soil_model_cols + proportion_logit_z_cols

print("建議放入模型的 X 變數：")
for c in model_x_cols:
    print(c)


# =========================================================
# 6) 輸出結果
# =========================================================

stats_df = pd.DataFrame(transform_stats)

with pd.ExcelWriter(output_path, engine="openpyxl") as writer:
    # df.to_excel(writer, sheet_name=f"{sheet_name}_transformed", index=False)


    pd.DataFrame({"model_x_cols": model_x_cols}).to_excel(
        writer,
        sheet_name="model_x_cols",
        index=False
    )

    stats_df.to_excel(
        writer,
        sheet_name="transform_stats",
        index=False
    )

stats_df.to_excel(stats_output_path, index=False)

print(f"\n轉換完成：{output_path}")
print(f"轉換參數表：{stats_output_path}")

建議放入模型的 X 變數：
該鄉鎮總人口數 (人數)_z
小於18歲人口數 (人數)_z
Income_regular (百萬元)_z
平均年齡（歲）_z
平均BMI  (kg/m2)_z
平均健保投保金額_z
Lead_water_z
PM2.5 (ug/m3)_z
每百萬人病床數 (床/萬人)_z
平均日常體能狀態（分）_z
加權 CCI_z
平均診斷到index date時間(日)_z
Lead_soil_ratio_logit_z
性別(男)_logit_z
癌症期別1_logit_z
癌症期別2_logit_z
仍吸菸_logit_z
仍喝酒_logit_z
癌症組織類型1_logit_z
癌症組織類型2_logit_z
癌症組織類型3_logit_z
癌症組織類型4_logit_z
OP指標日期前_logit_z
RT指標日期後_logit_z
併用藥物1_logit_z
併用藥物2_logit_z
併用藥物3_logit_z
併用藥物4_logit_z
併用藥物5_logit_z
併用藥物6_logit_z
併用藥物7_logit_z
併用藥物8_logit_z
併用藥物9_logit_z
併用藥物10_logit_z
併用藥物11_logit_z
醫學中心_logit_z

轉換完成：C:/Users/User/Desktop/LC analysis/result/回歸用資料.xlsx
轉換參數表：變數轉換參數表.xlsx


OLS回歸 對 死亡_cll 建模 僅協變數

In [ ]:
import numpy as np
import pandas as pd
import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor
from pathlib import Path

# =========================================================
# 0) 檔案路徑：若在自己電腦跑，請改成自己的路徑
# =========================================================
RAW_PATH = Path(r"C:/Users/User/Desktop/LC analysis/gdf_table_符合水痘_死亡資料變換可數型.xlsx")
RESULT_PATH = Path(r"C:/Users/User/Desktop/inflated_normal_full_individualized_sigma_results.xlsx")
OUT_PATH = Path(r"C:/Users/User/Desktop/merged_covariate_regression_results.xlsx")

# =========================================================
# 1) 讀取資料
# =========================================================
raw = pd.read_excel(RAW_PATH)
result = pd.read_excel(RESULT_PATH, sheet_name="row_level")

# =========================================================
# 2) key 清理：年份 + 鄉鎮名稱
#    注意：這不是變數維度轉換，只是為了合併資料
# =========================================================
def clean_key(df):
    df = df.copy()
    df["年份"] = pd.to_datetime(df["年份"], errors="coerce")
    df["year"] = df["年份"].dt.year
    df["鄉鎮_key"] = (
        df["鄉鎮"]
        .astype(str)
        .str.strip()
        .str.replace("臺", "台", regex=False)
        .str.replace(" ", "", regex=False)
    )
    return df

raw = clean_key(raw)
result = clean_key(result)

# 避免同一年同一鄉鎮重複導致 merge 後資料倍增
raw = raw.drop_duplicates(subset=["year", "鄉鎮_key"], keep="first")
result = result.drop_duplicates(subset=["year", "鄉鎮_key"], keep="first")

# =========================================================
# 3) 指定協變數
#    不做標準化、不做 logit、不做 cloglog、不做比例化
# =========================================================
covariates = [
    "該鄉鎮總人口數 (人數)",
    "小於18歲人口數 (人數)",   # 若不想放入模型，可註解掉這行
    "Lead_water",
    "Lead_soil",
    "PM2.5 (ug/m3)",
    "Income_regular (百萬元)",
    "每百萬人病床數 (床/萬人)",
    "平均年齡（歲）",
    "性別(男)",
    "平均BMI  (kg/m2)",
    "平均日常體能狀態（分）",
    "加權 CCI",
    "癌症期別1",
    "癌症期別2",
    "仍吸菸",
    "仍喝酒",
    "癌症組織類型1",
    "癌症組織類型2",
    "癌症組織類型3",
    "癌症組織類型4",
    "OP指標日期前",
    "RT指標日期後",
    "併用藥物1",
    "併用藥物2",
    "併用藥物3",
    "併用藥物4",
    "併用藥物5",
    "併用藥物6",
    "併用藥物7",
    "併用藥物8",
    "併用藥物9",
    "併用藥物10",
    "併用藥物11",
    "平均診斷到index date時間(日)",
    "平均健保投保金額",
    "醫學中心",
]

# 只保留實際存在於資料中的欄位
covariates = [c for c in covariates if c in raw.columns]

# =========================================================
# 4) 癌症期別與癌症組織類型的共線性處理
#    若期別1 + 期別2 約等於 1，兩者同時放入會與截距共線。
#    若組織類型1 + 2 + 3 + 4 約等於 1，也需拿掉一類作為 reference。
# =========================================================
reference_cols = [
    "癌症期別2",        # 癌症期別2 作為 reference group
    "癌症組織類型4",    # 癌症組織類型4 作為 reference group
]

reference_cols = [c for c in reference_cols if c in covariates]

x_cols = [c for c in covariates if c not in reference_cols]

# 不把死亡或模型輸出本身當 X，避免 outcome leakage
leakage_cols = [
    "死亡",
    "死亡_logit",
    "死亡_probit",
    "死亡_cll",
    "death_count",
    "death_count_raw",
    "y_true",
    "F_mean",
    "p_mean",
    "sigma_mean",
    "sigma2_mean",
    "yhat_mix_mean",
]

x_cols = [c for c in x_cols if c not in leakage_cols]

# =========================================================
# 5) 整合資料
#    result 是 BayesNF row_level 結果
#    raw 是原始協變數資料
# =========================================================
result_keep = [
    "year", "鄉鎮_key", "年份", "鄉鎮",
    "y_true",
    "F_mean",
    "p_mean",
    "sigma_mean",
    "sigma2_mean",
    "yhat_mix_mean",
    "err_mix",
    "abs_err_mix",
    "sq_err_mix",
    "pred_inflated_05",
    "true_inflated",
]

result_keep = [c for c in result_keep if c in result.columns]

raw_keep = [
    "year", "鄉鎮_key",
    "死亡",
    "死亡_cll",
    "死亡_logit",
    "死亡_probit",
    "death_count",
    "death_count_raw",
    "經度",
    "緯度",
] + covariates

raw_keep = list(dict.fromkeys([c for c in raw_keep if c in raw.columns]))

merged = result[result_keep].merge(
    raw[raw_keep],
    on=["year", "鄉鎮_key"],
    how="left",
    suffixes=("_result", "_raw")
)

not_matched = merged[merged[covariates].isna().all(axis=1)] if covariates else pd.DataFrame()

print(f"合併後資料列數: {len(merged)}")
print(f"未成功對到 raw 協變數的列數: {len(not_matched)}")

# =========================================================
# 6) 數值欄位整理
#    只是把 Excel 欄位轉 numeric，沒有做尺度轉換
# =========================================================
numeric_check_cols = list(
    set(
        x_cols
        + [
            "死亡",
            "死亡_cll",
            "y_true",
            "F_mean",
            "p_mean",
            "sigma_mean",
            "sigma2_mean",
            "yhat_mix_mean",
        ]
    )
)

for col in numeric_check_cols:
    if col in merged.columns:
        merged[col] = pd.to_numeric(merged[col], errors="coerce")

# 移除完全沒有變異的 X 欄位
x_cols = [
    c for c in x_cols
    if c in merged.columns and merged[c].nunique(dropna=True) > 1
]

# # =========================================================
# # 7) VIF 檢查函數
# # =========================================================
# def compute_vif(df, x_cols):
#     d = df[x_cols].replace([np.inf, -np.inf], np.nan).dropna().copy()

#     if d.empty or len(x_cols) < 2:
#         return pd.DataFrame(columns=["variable", "VIF"])

#     cols = [c for c in x_cols if d[c].nunique(dropna=True) > 1]
#     d = d[cols]

#     vif_rows = []
#     X = sm.add_constant(d, has_constant="add")

#     for i, col in enumerate(X.columns):
#         if col == "const":
#             continue

#         try:
#             vif = variance_inflation_factor(X.values, i)
#         except Exception:
#             vif = np.nan

#         vif_rows.append({
#             "variable": col,
#             "VIF": vif
#         })

#     return pd.DataFrame(vif_rows).sort_values("VIF", ascending=False)

# =========================================================
# 8) OLS 回歸函數
#    cov_type="HC3" 使用 robust standard error
# =========================================================
def fit_ols(df, y_col, x_cols):
    use_cols = [y_col] + x_cols
    use_cols = [c for c in use_cols if c in df.columns]

    d = df[use_cols].replace([np.inf, -np.inf], np.nan).dropna().copy()

    if d.empty:
        raise ValueError(f"{y_col}: 沒有可用資料，請檢查缺失值或欄位名稱。")

    if d[y_col].nunique(dropna=True) <= 1:
        raise ValueError(f"{y_col}: y 沒有變異，無法回歸。")

    xs = [
        c for c in x_cols
        if c in d.columns and d[c].nunique(dropna=True) > 1
    ]

    X = sm.add_constant(d[xs], has_constant="add")
    y = d[y_col]

    model = sm.OLS(y, X).fit(cov_type="HC3")

    coef_table = pd.DataFrame({
        "target": y_col,
        "variable": model.params.index,
        "coef": model.params.values,
        "std_err_HC3": model.bse.values,
        "t_value": model.tvalues.values,
        "p_value": model.pvalues.values,
        "ci_low": model.conf_int().iloc[:, 0].values,
        "ci_high": model.conf_int().iloc[:, 1].values,
    })

    fit_table = pd.DataFrame([{
        "target": y_col,
        "n_used": int(model.nobs),
        "n_predictors": len(xs),
        "R2": model.rsquared,
        "Adj_R2": model.rsquared_adj,
        "AIC": model.aic,
        "BIC": model.bic,
        "reference_stage_dropped": "癌症期別2" if "癌症期別2" in reference_cols else None,
        "reference_histology_dropped": "癌症組織類型4" if "癌症組織類型4" in reference_cols else None,
    }])

    return model, coef_table, fit_table
# =========================================================
# 9) 只針對 死亡_cll 做 OLS 回歸
#     模型：
#     死亡_cll_it = beta_0 + beta' X_it + epsilon_it
# =========================================================

target_col = "死亡_cll"

# 確認 y 存在
if target_col not in merged.columns:
    raise ValueError(f"找不到應變數欄位：{target_col}")

# 整理 y 與 X
use_cols = [target_col] + x_cols
use_cols = [c for c in use_cols if c in merged.columns]

reg_df = merged[use_cols].replace([np.inf, -np.inf], np.nan).dropna().copy()

# 移除沒有變異的 X
final_x_cols = [
    c for c in x_cols
    if c in reg_df.columns and reg_df[c].nunique(dropna=True) > 1
]

# y
y = reg_df[target_col]

# X，加上截距項 beta_0
X = reg_df[final_x_cols]
X = sm.add_constant(X, has_constant="add")

# OLS 估計
ols_model = sm.OLS(y, X).fit(cov_type="HC3")

# 顯示完整估計結果
print(ols_model.summary())

# =========================================================
# 10) 整理 beta 估計表
# =========================================================

beta_table = pd.DataFrame({
    "variable": ols_model.params.index,
    "estimate": ols_model.params.values,
    "std_error_HC3": ols_model.bse.values,
    "t_value": ols_model.tvalues.values,
    "p_value": ols_model.pvalues.values,
    "ci_low": ols_model.conf_int().iloc[:, 0].values,
    "ci_high": ols_model.conf_int().iloc[:, 1].values,
})

# 加上顯著性標記
beta_table["sig"] = pd.cut(
    beta_table["p_value"],
    bins=[-np.inf, 0.001, 0.01, 0.05, 0.10, np.inf],
    labels=["***", "**", "*", ".", ""]
)

# 模型摘要
model_fit_table = pd.DataFrame([{
    "model": "OLS",
    "formula": "死亡_cll_it = beta_0 + beta'X_it + epsilon_it",
    "target": target_col,
    "n_used": int(ols_model.nobs),
    "n_predictors": len(final_x_cols),
    "R2": ols_model.rsquared,
    "Adj_R2": ols_model.rsquared_adj,
    "AIC": ols_model.aic,
    "BIC": ols_model.bic,
    "reference_stage_dropped": "癌症期別2" if "癌症期別2" in reference_cols else None,
    "reference_histology_dropped": "癌症組織類型4" if "癌症組織類型4" in reference_cols else None,
}])

# =========================================================
# 11) 顯示結果
# =========================================================

print("\n模型摘要：")
print(model_fit_table)

print("\nBeta 估計表：")
print(beta_table)

# 顯著變數
sig_beta_table = beta_table[
    (beta_table["variable"] != "const") &
    (beta_table["p_value"] < 0.10)
].copy()

print("\n顯著變數 p < 0.10：")
print(sig_beta_table)

# =========================================================
# 12) 輸出 Excel
# =========================================================

OUT_PATH = Path(r"C:/Users/User/Desktop/death_cll_covariate_regression_results.xlsx")

with pd.ExcelWriter(OUT_PATH, engine="openpyxl") as writer:
    merged.to_excel(writer, sheet_name="merged_data", index=False)
    reg_df.to_excel(writer, sheet_name="regression_used_data", index=False)
    model_fit_table.to_excel(writer, sheet_name="model_fit", index=False)
    beta_table.to_excel(writer, sheet_name="beta_estimates", index=False)
    sig_beta_table.to_excel(writer, sheet_name="significant_p_lt_010", index=False)

print(f"\n已輸出：{OUT_PATH}")


合併後資料列數: 1297
未成功對到 raw 協變數的列數: 0
                            OLS Regression Results                            
Dep. Variable:                 死亡_cll   R-squared:                       0.122
Model:                            OLS   Adj. R-squared:                  0.098
Method:                 Least Squares   F-statistic:                     4.773
Date:                Sat, 16 May 2026   Prob (F-statistic):           1.81e-16
Time:                        16:37:15   Log-Likelihood:                -2913.0
No. Observations:                1290   AIC:                             5896.
Df Residuals:                    1255   BIC:                             6077.
Df Model:                          34                                         
Covariance Type:                  HC3                                         
                           coef    std err          z      P>|z|      [0.025      0.975]
---------------------------------------------------------------------------------------

c:\Users\User\.conda\envs\bayesnf310\lib\site-packages\statsmodels\base\model.py:1894: ValueWarning: covariance of constraints does not have full rank. The number of constraints is 34, but rank is 32
  warnings.warn('covariance of constraints does not have full '



已輸出：C:\Users\User\Desktop\death_cll_covariate_regression_results.xlsx


OLS回歸 對 死亡_cll 建模 F,sigma,協變數 連續變數標準化

In [2]:
import numpy as np
import pandas as pd
import statsmodels.api as sm
from scipy.optimize import minimize
from pathlib import Path

# =========================================================
# 0) 檔案路徑
# =========================================================
DATA_PATH = Path(r"C:/Users/User/Desktop/LC analysis/result/inflated_normal_highrisk_five_features_results.xlsx")
OUT_PATH = Path(r"C:/Users/User/Desktop/LC analysis/result/highrisk_five_features_three_regression_results_logit_prop.xlsx")

SHEET_NAME = "row_level"

# =========================================================
# 1) 讀取資料
# =========================================================
df = pd.read_excel(DATA_PATH, sheet_name=SHEET_NAME)

print("資料筆數：", len(df))
print("資料欄位數：", df.shape[1])

# =========================================================
# 2) 年份與鄉鎮 key 整理
# =========================================================
def parse_year(s):
    s = pd.Series(s).copy()
    numeric = pd.to_numeric(s, errors="coerce")

    out = pd.Series(index=s.index, dtype="float")

    mask_excel_date = numeric.between(30000, 60000)
    out.loc[mask_excel_date] = pd.to_datetime(
        numeric.loc[mask_excel_date],
        unit="D",
        origin="1899-12-30",
        errors="coerce"
    ).dt.year

    mask_year = numeric.between(1900, 2100)
    out.loc[mask_year] = numeric.loc[mask_year]

    mask_rest = out.isna()
    out.loc[mask_rest] = pd.to_datetime(
        s.loc[mask_rest],
        errors="coerce"
    ).dt.year

    return out.astype("Int64")


def clean_town(x):
    return (
        x.astype(str)
        .str.strip()
        .str.replace("臺", "台", regex=False)
        .str.replace(" ", "", regex=False)
    )


df = df.copy()

if "年份" in df.columns:
    df["year"] = parse_year(df["年份"])

if "鄉鎮" in df.columns:
    df["鄉鎮_key"] = clean_town(df["鄉鎮"])
    df["鄉鎮6"] = df["鄉鎮_key"].str[:6]

# =========================================================
# 3) 指定 X_it 協變數
# =========================================================
covariates = [
    "該鄉鎮總人口數 (人數)",
    # "小於18歲人口數 (人數)",
    "Lead_water",
    "Lead_soil",
    "PM2.5 (ug/m3)",
    "Income_regular (百萬元)",
    "每百萬人病床數 (床/萬人)",
    "平均年齡（歲）",
    "性別(男)",
    "平均BMI  (kg/m2)",
    "平均日常體能狀態（分）",
    "加權 CCI",
    "癌症期別1",
    "癌症期別2",
    "仍吸菸",
    "仍喝酒",
    "癌症組織類型1",
    "癌症組織類型2",
    "癌症組織類型3",
    "癌症組織類型4",
    "OP指標日期前",
    "RT指標日期後",
    "併用藥物1",
    "併用藥物2",
    "併用藥物3",
    "併用藥物4",
    "併用藥物5",
    "併用藥物6",
    "併用藥物7",
    "併用藥物8",
    "併用藥物9",
    "併用藥物10",
    "併用藥物11",
    "平均診斷到index date時間(日)",
    "平均健保投保金額",
    "醫學中心",
]

covariates = [c for c in covariates if c in df.columns]

# =========================================================
# 4) 共線性處理
#    類別變數移除一組作為 reference
# =========================================================
reference_cols = [
    "癌症期別2",
    "癌症組織類型4",
]

reference_cols = [c for c in reference_cols if c in covariates]

x_cols = [c for c in covariates if c not in reference_cols]

# 避免 outcome 或 BayesNF 輸出被放進一般 X
leakage_cols = [
    "死亡",
    "死亡_cll",
    "死亡_logit",
    "死亡_probit",
    "death_count",
    "death_count_raw",
    "y_true",
    "F_mean",
    "p_mean",
    "sigma_mean",
    "sigma2_mean",
    "F_q05",
    "F_q95",
    "p_q05",
    "p_q95",
    "sigma_q05",
    "sigma_q95",
    "sigma2_q05",
    "sigma2_q95",
    "yhat_mix_mean",
    "yhat_mix_q05",
    "yhat_mix_q95",
    "err_mix",
    "abs_err_mix",
    "sq_err_mix",
    "err_F_only",
    "abs_err_F_only",
    "sq_err_F_only",
    "pred_inflated_05",
    "true_inflated",
]

x_cols = [c for c in x_cols if c not in leakage_cols]

# =========================================================
# 5) 指定反應變數、BayesNF 平均場與變異場
# =========================================================
target_col = "死亡_cll"

F_COLS = ["F_mean"]
VAR_COLS = ["sigma2_mean"]

if target_col not in df.columns:
    raise ValueError(f"資料缺少反應變數欄位：{target_col}")

if "sigma2_mean" not in df.columns and "sigma_mean" in df.columns:
    df["sigma2_mean"] = pd.to_numeric(df["sigma_mean"], errors="coerce") ** 2

for c in F_COLS + VAR_COLS:
    if c not in df.columns:
        raise ValueError(f"資料缺少必要欄位：{c}")

# =========================================================
# 6) 數值欄位整理
# =========================================================
all_numeric_cols = [target_col] + x_cols + F_COLS + VAR_COLS

for c in all_numeric_cols:
    if c in df.columns:
        df[c] = pd.to_numeric(df[c], errors="coerce")

# 移除沒有變異的 X
x_cols = [
    c for c in x_cols
    if c in df.columns and df[c].nunique(dropna=True) > 1
]

# =========================================================
# 7) 指定連續型變數
#
# 這些變數會直接做：
# x -> z-score
#
# 其他沒有列在 STANDARDIZE_COLS 的變數，一律視為比例變數：
# p -> logit(p) -> z-score
# =========================================================
STANDARDIZE_COLS = [
    "該鄉鎮總人口數 (人數)",
    "Lead_water",
    "Lead_soil",
    "PM2.5 (ug/m3)",
    "Income_regular (百萬元)",
    "每百萬人病床數 (床/萬人)",
    "平均年齡（歲）",
    "性別(男)",
    "平均BMI  (kg/m2)",
    "平均日常體能狀態（分）",
    "加權 CCI",
    "平均診斷到index date時間(日)",
    "平均健保投保金額",
]

continuous_cols = [
    c for c in STANDARDIZE_COLS
    if c in x_cols and c in df.columns
]

proportion_cols = [
    c for c in x_cols
    if c not in continuous_cols
]

# =========================================================
# 8) 建立實際進入回歸樣本
# =========================================================
use_cols_raw = [target_col] + x_cols + F_COLS + VAR_COLS

id_cols = [
    c for c in ["年份", "year", "鄉鎮", "鄉鎮_key", "鄉鎮6"]
    if c in df.columns
]

reg_df = (
    df[id_cols + use_cols_raw]
    .replace([np.inf, -np.inf], np.nan)
    .dropna(subset=use_cols_raw)
    .copy()
)

print("實際進入回歸資料筆數：", len(reg_df))

# =========================================================
# 9) 變數轉換函數
# =========================================================
EPS = 1e-6

def zscore_series(s):
    s = pd.to_numeric(s, errors="coerce")
    mean_s = s.mean()
    sd_s = s.std(ddof=0)

    if pd.isna(sd_s) or sd_s <= 0:
        return None, mean_s, sd_s

    return (s - mean_s) / sd_s, mean_s, sd_s


def proportion_to_01(s, var_name):
    """
    將比例變數轉成 [0, 1] 尺度。

    可接受：
    1. 原本就是比例，例如 0.32
    2. 百分比格式，例如 32.0，會自動除以 100

    若數值不在 [0,1] 或 [0,100]，會報錯提醒。
    """
    s = pd.to_numeric(s, errors="coerce")

    min_s = s.min(skipna=True)
    max_s = s.max(skipna=True)

    if pd.isna(min_s) or pd.isna(max_s):
        raise ValueError(f"{var_name} 無法判斷比例範圍，可能全為缺失值。")

    if min_s >= 0 and max_s <= 1:
        p = s.copy()
        scale_type = "0_to_1"

    elif min_s >= 0 and max_s <= 100:
        p = s / 100
        scale_type = "0_to_100_divided_by_100"

    else:
        raise ValueError(
            f"{var_name} 被設定為比例變數，但數值範圍為 "
            f"[{min_s}, {max_s}]，不在 [0,1] 或 [0,100] 內。"
        )

    return p, scale_type


def safe_logit(p, eps=1e-6):
    """
    避免比例 p = 0 或 p = 1 時 logit 變成無限大。
    """
    p_clip = p.clip(lower=eps, upper=1 - eps)
    logit_value = np.log(p_clip / (1 - p_clip))

    n_lower_clip = int((p <= eps).sum())
    n_upper_clip = int((p >= 1 - eps).sum())

    return logit_value, p_clip, n_lower_clip, n_upper_clip


# =========================================================
# 10) 在 regression sample 內做變數轉換
#
# A. 指定連續型：
#    x -> z-score
#
# B. 其他比例型：
#    p -> logit(p) -> z-score
#
# 標準化的平均數與標準差都以實際進入模型的樣本計算。
# =========================================================
standardization_params = []

continuous_z_cols = []
proportion_logit_z_cols = []

# A) 指定連續型變數：直接 z-score
for c in continuous_cols:
    z_col = f"{c}_z"

    z_value, mean_c, sd_c = zscore_series(reg_df[c])

    if z_value is not None:
        reg_df[z_col] = z_value
        continuous_z_cols.append(z_col)

        standardization_params.append({
            "raw_variable": c,
            "variable_type": "specified_continuous",
            "transformation_type": "z-score",
            "proportion_scale_type": "",
            "intermediate_prop01_variable": "",
            "intermediate_logit_variable": "",
            "standardized_variable": z_col,
            "mean_used_for_z": mean_c,
            "sd_used_for_z": sd_c,
            "epsilon_for_logit": np.nan,
            "n_lower_clipped": np.nan,
            "n_upper_clipped": np.nan,
            "interpretation": "coefficient = change in 死亡_cll for one SD increase in this continuous covariate",
        })
    else:
        print(f"連續變數標準化略過：{c}，因為標準差為 0 或無法計算。")


# B) 非指定連續型變數：視為比例變數，先 logit 再 z-score
for c in proportion_cols:
    prop_col = f"{c}_prop01"
    logit_col = f"{c}_logit"
    logit_z_col = f"{c}_logit_z"

    p_raw, scale_type = proportion_to_01(reg_df[c], c)
    logit_value, p_clip, n_lower_clip, n_upper_clip = safe_logit(p_raw, eps=EPS)

    reg_df[prop_col] = p_clip
    reg_df[logit_col] = logit_value

    z_value, mean_logit, sd_logit = zscore_series(reg_df[logit_col])

    if z_value is not None:
        reg_df[logit_z_col] = z_value
        proportion_logit_z_cols.append(logit_z_col)

        standardization_params.append({
            "raw_variable": c,
            "variable_type": "non_specified_continuous_treated_as_proportion",
            "transformation_type": "proportion -> logit -> z-score",
            "proportion_scale_type": scale_type,
            "intermediate_prop01_variable": prop_col,
            "intermediate_logit_variable": logit_col,
            "standardized_variable": logit_z_col,
            "mean_used_for_z": mean_logit,
            "sd_used_for_z": sd_logit,
            "epsilon_for_logit": EPS,
            "n_lower_clipped": n_lower_clip,
            "n_upper_clipped": n_upper_clip,
            "interpretation": "coefficient = change in 死亡_cll for one SD increase in logit-transformed proportion",
        })
    else:
        print(f"比例變數 logit 後標準化略過：{c}，因為 logit 後標準差為 0 或無法計算。")


standardization_table = pd.DataFrame(standardization_params)

# =========================================================
# 11) 平均模型真正使用的 X
#
# 指定連續型：使用 *_z
# 比例型：使用 *_logit_z
# =========================================================
x_cols_model = continuous_z_cols + proportion_logit_z_cols

# 三種模型設定
# 1. Base OLS: X only
# 2. Augmented OLS: X + F_mean
# 3. FGLS: X + F_mean, Var(epsilon) uses sigma2_mean
mean_cols = x_cols_model + F_COLS
var_cols = VAR_COLS

print("指定連續型 z-score 變數數量：", len(continuous_z_cols))
print("比例變數 logit-z 變數數量：", len(proportion_logit_z_cols))
print("Base OLS 變數數量：", len(x_cols_model))
print("Augmented / FGLS 平均模型變數數量：", len(mean_cols))
print("FGLS 變異模型變數：", var_cols)

# =========================================================
# 12) 建立 y, X, Z
# =========================================================
y = reg_df[target_col].astype(float)

X_base = sm.add_constant(
    reg_df[x_cols_model].astype(float),
    has_constant="add"
)

X_aug = sm.add_constant(
    reg_df[mean_cols].astype(float),
    has_constant="add"
)

Z_var = reg_df[var_cols].astype(float)

# =========================================================
# 13) 第一種回歸：Base OLS，只放 X_it
# =========================================================
base_ols = sm.OLS(y, X_base).fit(cov_type="HC3")

# =========================================================
# 14) 第二種回歸：Augmented OLS，放 X_it + F_it
# =========================================================
aug_ols = sm.OLS(y, X_aug).fit(cov_type="HC3")

# =========================================================
# 15) 第三種回歸：FGLS
#
# 平均模型：
# 死亡_cll_it = beta0 + X_it beta + F_it beta_F + epsilon_it
#
# 變異模型：
# Var(epsilon_it) = sigma0_sq + gamma * sigma2_it
# =========================================================
def estimate_variance_params(resid, Z):
    r2 = np.asarray(resid, dtype=float) ** 2
    Zmat = np.asarray(Z, dtype=float)

    sigma0_init = max(np.nanmedian(r2), 1e-6)
    gamma_init = np.repeat(0.1, Zmat.shape[1])
    theta_init = np.r_[sigma0_init, gamma_init]

    def objective(theta):
        sigma0_sq = theta[0]
        gamma = theta[1:]
        h = sigma0_sq + Zmat @ gamma
        h = np.maximum(h, 1e-8)
        return np.mean((r2 - h) ** 2)

    bounds = [(1e-8, None)] + [(0, None)] * Zmat.shape[1]

    opt = minimize(
        objective,
        theta_init,
        method="L-BFGS-B",
        bounds=bounds
    )

    theta_hat = opt.x
    sigma0_sq_hat = theta_hat[0]
    gamma_hat = theta_hat[1:]

    h_hat = sigma0_sq_hat + Zmat @ gamma_hat
    h_hat = np.maximum(h_hat, 1e-8)

    return sigma0_sq_hat, gamma_hat, h_hat, opt


current_model = aug_ols
prev_params = current_model.params.copy()

max_iter = 20
tol = 1e-8

iteration_records = []

for it in range(max_iter):
    resid = y - current_model.fittedvalues

    sigma0_sq_hat, gamma_hat, h_hat, opt = estimate_variance_params(
        resid=resid,
        Z=Z_var
    )

    weights = 1.0 / h_hat

    next_model = sm.WLS(
        y,
        X_aug,
        weights=weights
    ).fit(cov_type="HC3")

    diff = np.max(np.abs(next_model.params - prev_params))

    iteration_records.append({
        "iteration": it + 1,
        "max_beta_change": diff,
        "sigma0_sq": sigma0_sq_hat,
        "gamma_sigma2_mean": gamma_hat[0] if len(gamma_hat) == 1 else np.nan,
        "success": opt.success,
        "objective_value": opt.fun,
    })

    print(
        f"iter={it+1:02d}, "
        f"max beta change={diff:.6e}, "
        f"sigma0_sq={sigma0_sq_hat:.6f}, "
        f"gamma={gamma_hat}"
    )

    current_model = next_model
    prev_params = next_model.params.copy()

    if diff < tol:
        break

fgls_model = current_model
iteration_table = pd.DataFrame(iteration_records)

# 最後使用 FGLS 殘差重估一次變異模型
final_resid = y - fgls_model.fittedvalues

sigma0_sq_hat, gamma_hat, h_hat, opt = estimate_variance_params(
    resid=final_resid,
    Z=Z_var
)

reg_df["fitted_base_ols"] = base_ols.fittedvalues
reg_df["resid_base_ols"] = y - base_ols.fittedvalues

reg_df["fitted_aug_ols"] = aug_ols.fittedvalues
reg_df["resid_aug_ols"] = y - aug_ols.fittedvalues

reg_df["fitted_fgls"] = fgls_model.fittedvalues
reg_df["resid_fgls"] = final_resid
reg_df["h_hat"] = h_hat
reg_df["weight_hat"] = 1.0 / h_hat

# =========================================================
# 16) Gaussian log-likelihood under heteroskedastic variance
# =========================================================
def hetero_loglik(y, fitted, h):
    resid = np.asarray(y) - np.asarray(fitted)
    h = np.maximum(np.asarray(h), 1e-8)
    ll = -0.5 * np.sum(
        np.log(2 * np.pi) + np.log(h) + resid**2 / h
    )
    return ll


ll_fgls = hetero_loglik(y, fgls_model.fittedvalues, h_hat)

n = len(y)
k_mean = len(fgls_model.params)
k_var = 1 + len(var_cols)
k_total = k_mean + k_var

aic_fgls = 2 * k_total - 2 * ll_fgls
bic_fgls = np.log(n) * k_total - 2 * ll_fgls

# =========================================================
# 17) 整理 beta 估計表
# =========================================================
def coef_table(model, model_name):
    ci = model.conf_int()

    out = pd.DataFrame({
        "model": model_name,
        "variable": model.params.index,
        "estimate": model.params.values,
        "std_error_HC3": model.bse.values,
        "t_value": model.tvalues.values,
        "p_value": model.pvalues.values,
        "ci_low": ci.iloc[:, 0].values,
        "ci_high": ci.iloc[:, 1].values,
    })

    out["sig"] = pd.cut(
        out["p_value"],
        bins=[-np.inf, 0.001, 0.01, 0.05, 0.10, np.inf],
        labels=["***", "**", "*", ".", ""]
    )

    return out


coef_base = coef_table(
    base_ols,
    "Base OLS: continuous z + proportion logit-z"
)

coef_aug = coef_table(
    aug_ols,
    "Augmented OLS: continuous z + proportion logit-z + F_mean"
)

coef_fgls = coef_table(
    fgls_model,
    "FGLS: continuous z + proportion logit-z + F_mean, Var uses sigma2_mean"
)

coef_out = pd.concat(
    [coef_base, coef_aug, coef_fgls],
    ignore_index=True
)

# =========================================================
# 18) 整理 variance model 估計表
# =========================================================
variance_table = pd.DataFrame({
    "variance_term": ["sigma0_sq"] + var_cols,
    "estimate": [sigma0_sq_hat] + list(gamma_hat),
    "meaning": (
        ["baseline residual variance"]
        + [f"effect of {c} on residual variance" for c in var_cols]
    )
})

# =========================================================
# 19) 模型比較表
# =========================================================
model_fit_table = pd.DataFrame([
    {
        "model": "Base OLS: continuous z + proportion logit-z",
        "mean_structure": "死亡_cll_it = beta0 + X_cont_z beta + X_prop_logit_z beta + epsilon_it",
        "variance_structure": "constant variance, robust SE reported",
        "n": int(base_ols.nobs),
        "k_mean": len(base_ols.params),
        "k_var": 1,
        "R2": base_ols.rsquared,
        "Adj_R2": base_ols.rsquared_adj,
        "AIC": base_ols.aic,
        "BIC": base_ols.bic,
        "sigma0_sq": np.nan,
        "gamma_sigma2_mean": np.nan,
    },
    {
        "model": "Augmented OLS: continuous z + proportion logit-z + F_mean",
        "mean_structure": "死亡_cll_it = beta0 + X_cont_z beta + X_prop_logit_z beta + F_it beta_F + epsilon_it",
        "variance_structure": "constant variance, robust SE reported",
        "n": int(aug_ols.nobs),
        "k_mean": len(aug_ols.params),
        "k_var": 1,
        "R2": aug_ols.rsquared,
        "Adj_R2": aug_ols.rsquared_adj,
        "AIC": aug_ols.aic,
        "BIC": aug_ols.bic,
        "sigma0_sq": np.nan,
        "gamma_sigma2_mean": np.nan,
    },
    {
        "model": "FGLS: continuous z + proportion logit-z + F_mean, Var uses sigma2_mean",
        "mean_structure": "死亡_cll_it = beta0 + X_cont_z beta + X_prop_logit_z beta + F_it beta_F + epsilon_it",
        "variance_structure": "Var(epsilon_it) = sigma0_sq + gamma * sigma2_it",
        "n": int(fgls_model.nobs),
        "k_mean": k_mean,
        "k_var": k_var,
        "R2": fgls_model.rsquared,
        "Adj_R2": fgls_model.rsquared_adj,
        "AIC": aic_fgls,
        "BIC": bic_fgls,
        "sigma0_sq": sigma0_sq_hat,
        "gamma_sigma2_mean": gamma_hat[0] if len(gamma_hat) == 1 else np.nan,
    }
])

# =========================================================
# 20) F_mean 效果獨立整理
# =========================================================
F_effect_table = coef_out[
    coef_out["variable"].isin(F_COLS)
].copy()

# =========================================================
# 21) 顯著變數整理
# =========================================================
sig_out = coef_out[
    (coef_out["variable"] != "const") &
    (coef_out["p_value"] < 0.10)
].copy()

# =========================================================
# 22) 模型設定表
# =========================================================
variable_design_records = []

for c in continuous_z_cols:
    raw_name = c.replace("_z", "")
    variable_design_records.append({
        "variable_used_in_model": c,
        "raw_variable": raw_name,
        "role": "mean_model_continuous_X",
        "transformation": "x -> z-score",
        "standardized": True,
    })

for c in proportion_logit_z_cols:
    raw_name = c.replace("_logit_z", "")
    variable_design_records.append({
        "variable_used_in_model": c,
        "raw_variable": raw_name,
        "role": "mean_model_proportion_X",
        "transformation": "p -> logit(p) -> z-score",
        "standardized": True,
    })

for c in F_COLS:
    variable_design_records.append({
        "variable_used_in_model": c,
        "raw_variable": c,
        "role": "BayesNF_mean_field",
        "transformation": "none",
        "standardized": False,
    })

for c in VAR_COLS:
    variable_design_records.append({
        "variable_used_in_model": c,
        "raw_variable": c,
        "role": "variance_model_sigma2",
        "transformation": "none",
        "standardized": False,
    })

variable_design_table = pd.DataFrame(variable_design_records)

reference_table = pd.DataFrame({
    "removed_reference_variable": reference_cols,
    "reason": "reference group to avoid perfect multicollinearity"
})

proportion_transform_table = standardization_table[
    standardization_table["variable_type"] == "non_specified_continuous_treated_as_proportion"
].copy()

continuous_transform_table = standardization_table[
    standardization_table["variable_type"] == "specified_continuous"
].copy()

# =========================================================
# 23) 輸出 Excel
# =========================================================
with pd.ExcelWriter(OUT_PATH, engine="openpyxl") as writer:
    df.to_excel(writer, sheet_name="original_row_level", index=False)
    reg_df.to_excel(writer, sheet_name="regression_used_data", index=False)
    model_fit_table.to_excel(writer, sheet_name="model_comparison", index=False)
    coef_out.to_excel(writer, sheet_name="beta_estimates", index=False)
    sig_out.to_excel(writer, sheet_name="significant_p_lt_010", index=False)
    F_effect_table.to_excel(writer, sheet_name="F_effect", index=False)
    variance_table.to_excel(writer, sheet_name="variance_gamma", index=False)
    iteration_table.to_excel(writer, sheet_name="fgls_iterations", index=False)
    standardization_table.to_excel(writer, sheet_name="standardization_params", index=False)
    continuous_transform_table.to_excel(writer, sheet_name="continuous_z_params", index=False)
    proportion_transform_table.to_excel(writer, sheet_name="proportion_logit_z_params", index=False)
    variable_design_table.to_excel(writer, sheet_name="variable_design", index=False)
    reference_table.to_excel(writer, sheet_name="reference_cols", index=False)

print("\n==============================")
print("Base OLS summary")
print("==============================")
print(base_ols.summary())

print("\n==============================")
print("Augmented OLS summary: continuous z + proportion logit-z + F_mean")
print("==============================")
print(aug_ols.summary())

print("\n==============================")
print("FGLS summary: continuous z + proportion logit-z + F_mean, variance uses sigma2_mean")
print("==============================")
print(fgls_model.summary())

print("\n==============================")
print("Variance model")
print("==============================")
print(variance_table)

print("\n==============================")
print("Model comparison")
print("==============================")
print(model_fit_table)

print("\n==============================")
print("Continuous z variables")
print("==============================")
print(continuous_z_cols)

print("\n==============================")
print("Proportion logit-z variables")
print("==============================")
print(proportion_logit_z_cols)

print(f"\n已輸出：{OUT_PATH}")

資料筆數： 1297
資料欄位數： 90
實際進入回歸資料筆數： 1290
指定連續型 z-score 變數數量： 13
比例變數 logit-z 變數數量： 20
Base OLS 變數數量： 33
Augmented / FGLS 平均模型變數數量： 34
FGLS 變異模型變數： ['sigma2_mean']
iter=01, max beta change=9.325873e-15, sigma0_sq=4.060061, gamma=[0.]

Base OLS summary
                            OLS Regression Results                            
Dep. Variable:                 死亡_cll   R-squared:                       0.295
Model:                            OLS   Adj. R-squared:                  0.276
Method:                 Least Squares   F-statistic:                     23.74
Date:                Wed, 01 Jul 2026   Prob (F-statistic):          1.26e-108
Time:                        20:51:32   Log-Likelihood:                -2771.8
No. Observations:                1290   AIC:                             5612.
Df Residuals:                    1256   BIC:                             5787.
Df Model:                          33                                         
Covariance Type:                  HC3    

MIX model 對 死亡_cll 建模 F,sigma,協變數 連續變數標準化

In [6]:

import warnings
import numpy as np
import pandas as pd
import statsmodels.api as sm

from pathlib import Path
from scipy.optimize import minimize
from scipy.stats import norm
from statsmodels.tools.numdiff import approx_hess
from statsmodels.tools.sm_exceptions import PerfectSeparationError
from sklearn.metrics import roc_auc_score


# =========================================================
# 0) File paths
# =========================================================
DATA_FILE = Path(
    r"C:/Users/User/Desktop/LC analysis/result/inflated_normal_highrisk_five_features_results.xlsx"
)

OUT_FILE = Path(
    r"C:/Users/User/Desktop/LC analysis/result/inflated_normal_mixture_X_vs_XF_comparison.xlsx"
)

RESULT_SHEET = "row_level"


# =========================================================
# 1) Basic utilities
# =========================================================
def unique(seq):
    out, seen = [], set()
    for x in seq:
        if x not in seen:
            out.append(x)
            seen.add(x)
    return out


def sig_star(p):
    if pd.isna(p):
        return ""
    if p < 0.001:
        return "***"
    if p < 0.01:
        return "**"
    if p < 0.05:
        return "*"
    if p < 0.10:
        return "."
    return ""


def parse_year(s):
    s = pd.Series(s).copy()
    numeric = pd.to_numeric(s, errors="coerce")

    out = pd.Series(index=s.index, dtype="float")

    mask_excel_date = numeric.between(30000, 60000)
    out.loc[mask_excel_date] = pd.to_datetime(
        numeric.loc[mask_excel_date],
        unit="D",
        origin="1899-12-30",
        errors="coerce"
    ).dt.year

    mask_year = numeric.between(1900, 2100)
    out.loc[mask_year] = numeric.loc[mask_year]

    mask_rest = out.isna()
    out.loc[mask_rest] = pd.to_datetime(
        s.loc[mask_rest],
        errors="coerce"
    ).dt.year

    return out.astype("Int64")


def clean_town(x):
    return (
        x.astype(str)
        .str.strip()
        .str.replace("臺", "台", regex=False)
        .str.replace(" ", "", regex=False)
    )


def zscore_series(s):
    s = pd.to_numeric(s, errors="coerce")
    mean_s = s.mean()
    sd_s = s.std(ddof=0)

    if pd.isna(sd_s) or sd_s <= 0:
        return None, mean_s, sd_s

    return (s - mean_s) / sd_s, mean_s, sd_s


def safe_logit01(s, eps=1e-6):
    s = pd.to_numeric(s, errors="coerce")
    s_clip = np.clip(s, eps, 1 - eps)
    return np.log(s_clip / (1 - s_clip))


def proportion_to_01(s, var_name):
    """
    將比例變數轉成 [0,1] 尺度。

    可接受：
    1. 原本就是比例，例如 0.32
    2. 百分比格式，例如 32.0，會自動除以 100

    若數值不在 [0,1] 或 [0,100]，會報錯提醒。
    """
    s = pd.to_numeric(s, errors="coerce")

    min_s = s.min(skipna=True)
    max_s = s.max(skipna=True)

    if pd.isna(min_s) or pd.isna(max_s):
        raise ValueError(f"{var_name} 無法判斷比例範圍，可能全為缺失值。")

    if min_s >= 0 and max_s <= 1:
        p = s.copy()
        scale_type = "0_to_1"

    elif min_s >= 0 and max_s <= 100:
        p = s / 100
        scale_type = "0_to_100_divided_by_100"

    else:
        raise ValueError(
            f"{var_name} 被設定為比例變數，但數值範圍為 "
            f"[{min_s}, {max_s}]，不在 [0,1] 或 [0,100] 內。"
        )

    return p, scale_type


def safe_logit_with_clip_count(p, eps=1e-6):
    p = pd.to_numeric(p, errors="coerce")

    n_lower_clip = int((p <= eps).sum())
    n_upper_clip = int((p >= 1 - eps).sum())

    p_clip = p.clip(lower=eps, upper=1 - eps)
    logit_value = np.log(p_clip / (1 - p_clip))

    return logit_value, p_clip, n_lower_clip, n_upper_clip


def rmse(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    return float(np.sqrt(np.mean((y_true - y_pred) ** 2)))


def mae(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    return float(np.mean(np.abs(y_true - y_pred)))


def mean_impute_columns(df, cols):
    """
    對指定欄位先轉成 numeric，將 inf 轉成 NaN，
    再以該欄位平均數補值。

    這裡只處理模型會用到的數值欄位；
    id 欄位與 true_inflated 不在此處平均補值。
    若某欄位整欄都是缺失，沒有平均數可補，後續會視情況移除或停止。
    """
    records = []

    for col in unique([c for c in cols if c in df.columns]):
        s = pd.to_numeric(df[col], errors="coerce").replace([np.inf, -np.inf], np.nan)

        n_missing_before = int(s.isna().sum())
        mean_value = s.mean(skipna=True)

        if pd.notna(mean_value):
            df[col] = s.fillna(mean_value)
            status = "imputed_by_column_mean" if n_missing_before > 0 else "no_missing"
        else:
            df[col] = s
            status = "all_missing_not_imputed"

        records.append({
            "variable": col,
            "n_missing_before": n_missing_before,
            "mean_used_for_imputation": mean_value,
            "n_missing_after": int(pd.isna(df[col]).sum()),
            "status": status,
        })

    return pd.DataFrame(records)


def check_no_missing(df, cols, stage_name):
    """補值後的安全檢查：不再偷偷 drop row。"""
    cols = [c for c in unique(cols) if c in df.columns]
    missing = df[cols].isna().sum()
    missing = missing[missing > 0].sort_values(ascending=False)

    if len(missing) > 0:
        print(f"\n[{stage_name}] 補值後仍有缺失，請檢查下列欄位：")
        print(missing)
        raise ValueError(
            f"{stage_name}: 補值後仍有必要欄位缺失。"
            "程式已停止，避免用 dropna 偷偷刪除資料列。"
        )


# =========================================================
# 2) Read data
# =========================================================
df = pd.read_excel(DATA_FILE, sheet_name=RESULT_SHEET)
n_rows_original = len(df)

print("資料筆數：", n_rows_original)
print("資料欄位數：", df.shape[1])

# 若 Excel 讀進來有重複欄位名稱，可補正。
df = df.rename(columns={
    "癌症期別": "癌症期別1",
    "...17": "癌症期別2",
    "癌症組織類型": "癌症組織類型1",
    "...21": "癌症組織類型2",
    "...22": "癌症組織類型3",
    "...23": "癌症組織類型4",
})

df = df.copy()

if "年份" in df.columns:
    df["year"] = parse_year(df["年份"])

if "鄉鎮" in df.columns:
    df["鄉鎮_key"] = clean_town(df["鄉鎮"])
    df["鄉鎮6"] = df["鄉鎮_key"].str[:6]


# =========================================================
# 3) Define outcome, point mass, and BayesNF outputs
# =========================================================
POINT_MASS_C = -6.907255070523717

if "死亡_cll" in df.columns:
    Y_COL = "死亡_cll"
elif "y_imputed" in df.columns:
    Y_COL = "y_imputed"
elif "y_true" in df.columns:
    Y_COL = "y_true"
else:
    raise ValueError("資料缺少反應變數欄位：死亡_cll、y_imputed、y_true 都不存在。")

F_COL = "F_mean"
P_COL = "p_mean"

for c in [Y_COL, F_COL, P_COL]:
    if c not in df.columns:
        raise ValueError(f"資料缺少必要欄位：{c}")

df[Y_COL] = pd.to_numeric(df[Y_COL], errors="coerce")
df[F_COL] = pd.to_numeric(df[F_COL], errors="coerce")
df[P_COL] = pd.to_numeric(df[P_COL], errors="coerce")

if "true_inflated" not in df.columns:
    df["true_inflated"] = (
        np.abs(pd.to_numeric(df[Y_COL], errors="coerce") - POINT_MASS_C) < 1e-8
    ).astype(int)
else:
    df["true_inflated"] = pd.to_numeric(df["true_inflated"], errors="coerce").astype("Int64")


# =========================================================
# 4) Covariates
# =========================================================
covariates = [
    "該鄉鎮總人口數 (人數)",
    # "小於18歲人口數 (人數)",
    "Lead_water",
    "Lead_soil",
    "PM2.5 (ug/m3)",
    "Income_regular (百萬元)",
    "每百萬人病床數 (床/萬人)",
    "平均年齡（歲）",
    "性別(男)",
    "平均BMI  (kg/m2)",
    "平均日常體能狀態（分）",
    "加權 CCI",
    "癌症期別1",
    "癌症期別2",
    "仍吸菸",
    "仍喝酒",
    "癌症組織類型1",
    "癌症組織類型2",
    "癌症組織類型3",
    "癌症組織類型4",
    "OP指標日期前",
    "RT指標日期後",
    "併用藥物1",
    "併用藥物2",
    "併用藥物3",
    "併用藥物4",
    "併用藥物5",
    "併用藥物6",
    "併用藥物7",
    "併用藥物8",
    "併用藥物9",
    "併用藥物10",
    "併用藥物11",
    "平均診斷到index date時間(日)",
    "平均健保投保金額",
    "醫學中心",
]

covariates = [c for c in covariates if c in df.columns]

# 移除 reference category，避免完全共線性。
reference_cols = [
    "癌症期別2",
    "癌症組織類型4",
]
reference_cols = [c for c in reference_cols if c in covariates]

x_raw_cols = [c for c in covariates if c not in reference_cols]

# 避免 outcome 或 BayesNF 輸出被放進一般 X
leakage_cols = [
    "死亡",
    "死亡_cll",
    "死亡_logit",
    "死亡_probit",
    "death_count",
    "death_count_raw",
    "y_true",
    "y_imputed",
    "F_mean",
    "p_mean",
    "sigma_mean",
    "sigma2_mean",
    "F_q05",
    "F_q95",
    "p_q05",
    "p_q95",
    "sigma_q05",
    "sigma_q95",
    "sigma2_q05",
    "sigma2_q95",
    "yhat_mix_mean",
    "yhat_mix_q05",
    "yhat_mix_q95",
    "err_mix",
    "abs_err_mix",
    "sq_err_mix",
    "err_F_only",
    "abs_err_F_only",
    "sq_err_F_only",
    "pred_inflated_05",
    "true_inflated",
]

x_raw_cols = [c for c in x_raw_cols if c not in leakage_cols]

# 後續 variance equation 可能會用到的風險指標，也一起納入補值流程。
risk_cols = [
    c for c in ["high_risk", "potential_high_risk"]
    if c in df.columns
]

# =========================================================
# 5) Mean imputation before transformation
#
# 為避免 complete-case dropna 將資料刪掉，先對模型會用到的數值欄位補平均數。
# true_inflated 不用平均補值，因為它是 0/1 膨脹指標。
# =========================================================
impute_cols = unique([Y_COL, F_COL, P_COL] + x_raw_cols + risk_cols)
imputation_table = mean_impute_columns(df, impute_cols)

# 必要欄位若整欄缺失，平均數無法補，必須停止。
mandatory_cols = [Y_COL, F_COL, P_COL]
all_missing_mandatory = imputation_table.loc[
    (imputation_table["variable"].isin(mandatory_cols))
    & (imputation_table["status"] == "all_missing_not_imputed"),
    "variable"
].tolist()
if len(all_missing_mandatory) > 0:
    raise ValueError(f"必要欄位整欄缺失，無法用欄位平均補值：{all_missing_mandatory}")

# 選入模型的 optional covariates 若整欄缺失，沒有平均數可補，直接從模型變數中移除。
# 這種情況不是刪資料列，而是移除無資訊欄位。
all_missing_optional = imputation_table.loc[
    (~imputation_table["variable"].isin(mandatory_cols))
    & (imputation_table["status"] == "all_missing_not_imputed"),
    "variable"
].tolist()

if len(all_missing_optional) > 0:
    print("下列 optional 欄位整欄缺失，無法平均補值，已從模型變數移除：")
    print(all_missing_optional)

x_raw_cols = [c for c in x_raw_cols if c not in all_missing_optional]
risk_cols = [c for c in risk_cols if c not in all_missing_optional]

# 若 true_inflated 有缺失，改用補值後的 Y_COL 是否等於膨脹點判定。
df["true_inflated"] = pd.to_numeric(df["true_inflated"], errors="coerce")
missing_inflated = df["true_inflated"].isna()
if missing_inflated.any():
    df.loc[missing_inflated, "true_inflated"] = (
        np.abs(df.loc[missing_inflated, Y_COL] - POINT_MASS_C) < 1e-8
    ).astype(int)

check_no_missing(df, mandatory_cols + ["true_inflated"], "完成平均補值後的必要欄位檢查")
df["true_inflated"] = df["true_inflated"].astype(int)

print("平均補值欄位數：", len(imputation_table))
print("平均補值總格數：", int(imputation_table["n_missing_before"].sum()))
print("補值後總資料筆數：", len(df))

# 移除沒有變異的 X。這是移除無資訊欄位，不是刪資料列。
x_raw_cols = [
    c for c in x_raw_cols
    if c in df.columns and df[c].nunique(dropna=True) > 1
]


# =========================================================
# 6) Variable type design
#
# Specified continuous variables:
# x -> z-score
#
# Other variables:
# p -> logit(p) -> z-score
# =========================================================
CONTINUOUS_COLS = [
    "該鄉鎮總人口數 (人數)",
    "Lead_water",
    "Lead_soil",
    "PM2.5 (ug/m3)",
    "Income_regular (百萬元)",
    "每百萬人病床數 (床/萬人)",
    "平均年齡（歲）",
    "性別(男)",
    "平均BMI  (kg/m2)",
    "平均日常體能狀態（分）",
    "加權 CCI",
    "平均診斷到index date時間(日)",
    "平均健保投保金額",
]

continuous_cols = [
    c for c in CONTINUOUS_COLS
    if c in x_raw_cols and c in df.columns
]

proportion_cols = [
    c for c in x_raw_cols
    if c not in continuous_cols
]


# =========================================================
# 7) Build base data
# =========================================================
id_cols = [
    c for c in ["年份", "year", "鄉鎮", "鄉鎮_key", "鄉鎮6"]
    if c in df.columns
]

needed_raw = unique(
    id_cols
    + [Y_COL, "true_inflated", F_COL, P_COL]
    + x_raw_cols
    + risk_cols
)

tf = (
    df[needed_raw]
    .replace([np.inf, -np.inf], np.nan)
    .copy()
)

# 補值已在前面完成；此處不再 dropna。
# 若還有缺失，直接報錯列出欄位，避免資料列被安靜刪掉。
required_base_cols = [Y_COL, "true_inflated", F_COL, P_COL] + x_raw_cols + risk_cols
check_no_missing(tf, required_base_cols, "轉換前資料檢查")
tf["true_inflated"] = tf["true_inflated"].astype(int)

print("轉換前資料筆數（補值後，未刪列）：", len(tf))
print("膨脹點筆數：", int(tf["true_inflated"].sum()))
print("非膨脹點筆數：", int((tf["true_inflated"] == 0).sum()))


# =========================================================
# 8) Transform BNF p_mean
# =========================================================
tf["logit_p_BNF"] = safe_logit01(tf[P_COL], eps=1e-6)


# =========================================================
# 9) Transform covariates
# =========================================================
transform_records = []
transformed_cols = {}

continuous_z_cols = []
proportion_logit_z_cols = []

# A) Specified continuous: x -> z-score
for c in continuous_cols:
    z_col = f"{c}_z"

    z_value, mean_c, sd_c = zscore_series(tf[c])

    if z_value is not None:
        tf[z_col] = z_value
        transformed_cols[c] = z_col
        continuous_z_cols.append(z_col)

        transform_records.append({
            "raw_variable": c,
            "model_variable": z_col,
            "variable_type": "specified_continuous",
            "transformation": "x -> z-score",
            "proportion_scale_type": "",
            "mean_used_for_z": mean_c,
            "sd_used_for_z": sd_c,
            "epsilon_for_logit": np.nan,
            "n_lower_clipped": np.nan,
            "n_upper_clipped": np.nan,
        })
    else:
        print(f"連續變數標準化略過：{c}，因為標準差為 0 或無法計算。")


# B) Other variables: p -> logit(p) -> z-score
for c in proportion_cols:
    prop_col = f"{c}_prop01"
    logit_col = f"{c}_logit"
    logit_z_col = f"{c}_logit_z"

    p_raw, scale_type = proportion_to_01(tf[c], c)
    logit_value, p_clip, n_lower_clip, n_upper_clip = safe_logit_with_clip_count(
        p_raw,
        eps=1e-6
    )

    tf[prop_col] = p_clip
    tf[logit_col] = logit_value

    z_value, mean_logit, sd_logit = zscore_series(tf[logit_col])

    if z_value is not None:
        tf[logit_z_col] = z_value
        transformed_cols[c] = logit_z_col
        proportion_logit_z_cols.append(logit_z_col)

        transform_records.append({
            "raw_variable": c,
            "model_variable": logit_z_col,
            "variable_type": "non_specified_continuous_treated_as_proportion",
            "transformation": "p -> logit(p) -> z-score",
            "proportion_scale_type": scale_type,
            "mean_used_for_z": mean_logit,
            "sd_used_for_z": sd_logit,
            "epsilon_for_logit": 1e-6,
            "n_lower_clipped": n_lower_clip,
            "n_upper_clipped": n_upper_clip,
        })
    else:
        print(f"比例變數 logit 後標準化略過：{c}，因為 logit 後標準差為 0 或無法計算。")


transform_map = pd.DataFrame(transform_records)

print("指定連續型 z-score 變數數量：", len(continuous_z_cols))
print("比例變數 logit-z 變數數量：", len(proportion_logit_z_cols))


# =========================================================
# 10) Define mean equation and variance equation variables
# =========================================================
# MIX-X: only specified transformed covariates
X_ONLY_VARS = unique(
    continuous_z_cols
    + proportion_logit_z_cols
)

# MIX-XF: specified transformed covariates + BayesNF F_it
X_F_VARS = unique(
    [F_COL]
    + continuous_z_cols
    + proportion_logit_z_cols
)

# Variance equation:
# keep same V structure for both models.
V_raw_candidates = [
    "Lead_water",
    "Lead_soil",
    "PM2.5 (ug/m3)",
    "每百萬人病床數 (床/萬人)",
    "平均年齡（歲）",
    "平均日常體能狀態（分）",
    "加權 CCI",
    "平均診斷到index date時間(日)",
]

V_vars = []
for raw_c in V_raw_candidates:
    if raw_c in transformed_cols:
        V_vars.append(transformed_cols[raw_c])

for risk_col in ["high_risk", "potential_high_risk"]:
    if risk_col in tf.columns:
        tf[risk_col] = pd.to_numeric(tf[risk_col], errors="coerce")
        if tf[risk_col].nunique(dropna=True) > 1:
            V_vars.append(risk_col)

V_vars = unique([v for v in V_vars if v in tf.columns])

needed_model = unique(
    [Y_COL, "true_inflated", P_COL, "logit_p_BNF"]
    + X_ONLY_VARS
    + X_F_VARS
    + V_vars
)

dat = (
    tf[needed_model]
    .replace([np.inf, -np.inf], np.nan)
    .copy()
)

# 不再使用 dropna。轉換後若有 NaN，代表某個轉換或模型變數仍有問題。
check_no_missing(dat, needed_model, "進入 mixture model 前檢查")
dat["true_inflated"] = dat["true_inflated"].astype(int)

print("進入 mixture model 的資料筆數（未刪列）：", len(dat))
print("進入 mixture model 的膨脹點筆數：", int(dat["true_inflated"].sum()))
print("進入 mixture model 的非膨脹點筆數：", int((dat["true_inflated"] == 0).sum()))
print("MIX-X mean variables 數量：", len(X_ONLY_VARS))
print("MIX-XF mean variables 數量：", len(X_F_VARS))
print("Variance equation V_vars 數量：", len(V_vars))


# =========================================================
# 11) q-model: alpha_p, beta_p
# =========================================================
def fit_q_model(dat, p_col=P_COL):
    y_bin = dat["true_inflated"].astype(int).values
    Xq = sm.add_constant(dat[["logit_p_BNF"]], has_constant="add")

    q_note = ""
    q_complete_separation = False
    q_usable_for_significance = True

    try:
        with warnings.catch_warnings(record=True) as w:
            warnings.simplefilter("always")
            q_res = sm.Logit(y_bin, Xq).fit(disp=False, maxiter=200)
            q_note = "; ".join(sorted(set([str(ww.message) for ww in w])))

        q_params = q_res.params
        q_bse = q_res.bse
        q_z = q_res.tvalues
        q_pvalues = q_res.pvalues

    except PerfectSeparationError as e:
        q_complete_separation = True
        q_usable_for_significance = False
        q_note = f"PerfectSeparationError: {str(e)}"

        glm_binom = sm.GLM(y_bin, Xq, family=sm.families.Binomial())
        q_res_reg = glm_binom.fit_regularized(alpha=1e-6, L1_wt=0.0, maxiter=1000)

        q_params = pd.Series(q_res_reg.params, index=Xq.columns)
        q_bse = pd.Series(np.nan, index=Xq.columns)
        q_z = pd.Series(np.nan, index=Xq.columns)
        q_pvalues = pd.Series(np.nan, index=Xq.columns)

    except Exception as e:
        q_complete_separation = True
        q_usable_for_significance = False
        q_note = f"Logit failed and fallback used: {str(e)}"

        glm_binom = sm.GLM(y_bin, Xq, family=sm.families.Binomial())
        q_res_reg = glm_binom.fit_regularized(alpha=1e-6, L1_wt=0.0, maxiter=1000)

        q_params = pd.Series(q_res_reg.params, index=Xq.columns)
        q_bse = pd.Series(np.nan, index=Xq.columns)
        q_z = pd.Series(np.nan, index=Xq.columns)
        q_pvalues = pd.Series(np.nan, index=Xq.columns)

    try:
        auc = roc_auc_score(y_bin, dat[p_col])
    except Exception:
        auc = np.nan

    threshold_errors = int(
        ((dat[p_col] > 0.5).astype(int).values != y_bin).sum()
    )

    if pd.notna(auc):
        q_complete_separation = (
            q_complete_separation
            or (auc == 1.0 and threshold_errors == 0)
            or ("Perfect separation" in q_note)
        )

    if q_complete_separation:
        q_usable_for_significance = False

    logit_q_hat = (
        q_params.get("const", 0.0)
        + q_params.get("logit_p_BNF", 0.0) * dat["logit_p_BNF"]
    )

    q_hat = 1 / (1 + np.exp(-np.clip(logit_q_hat, -30, 30)))
    q_hat = pd.Series(q_hat, index=dat.index, name="q_hat")

    eps = 1e-12
    q_loglik = np.sum(
        np.where(
            y_bin == 1,
            np.log(np.clip(q_hat.values, eps, 1 - eps)),
            np.log(np.clip(1 - q_hat.values, eps, 1 - eps))
        )
    )

    def q_row(parameter_name, variable_name, var_key):
        pval = float(q_pvalues.get(var_key, np.nan))

        return {
            "model": "Common q-model",
            "component": "inflation probability",
            "parameter": parameter_name,
            "equation_part": "q_it = logit^{-1}(alpha_p + beta_p logit(p_BNF))",
            "variable": variable_name,
            "coef": float(q_params.get(var_key, np.nan)),
            "se": float(q_bse.get(var_key, np.nan)),
            "z": float(q_z.get(var_key, np.nan)),
            "p_value": pval,
            "sig": "NA" if not q_usable_for_significance else sig_star(pval),
            "usable_for_significance": "No" if not q_usable_for_significance else "Yes",
            "note": (
                "complete separation or fallback model; Wald test not interpretable"
                if not q_usable_for_significance
                else q_note
            ),
        }

    q_rows = pd.DataFrame([
        q_row("alpha_p", "(Intercept)", "const"),
        q_row("beta_p", "logit_p_BNF", "logit_p_BNF"),
    ])

    q_info = {
        "q_model_AUC_p_mean": float(auc) if pd.notna(auc) else np.nan,
        "q_model_threshold_0.5_errors": int(threshold_errors),
        "q_model_complete_separation": str(q_complete_separation),
        "q_model_usable_for_significance": str(q_usable_for_significance),
        "q_model_loglik": float(q_loglik),
        "q_model_n_params": int(len(q_params)),
        "q_model_note": q_note,
    }

    return q_params, q_hat, q_loglik, q_rows, q_info


q_params, q_hat, q_loglik, q_rows, q_info = fit_q_model(dat, p_col=P_COL)
dat["q_hat"] = q_hat


# =========================================================
# 12) Fit normal component
# =========================================================
def fit_normal_component(
    dat,
    model_name,
    x_vars,
    v_vars,
    y_col,
    q_hat,
    q_loglik,
    point_mass_c,
):
    normal_dat = dat[dat["true_inflated"] == 0].copy()

    if len(normal_dat) <= 5:
        raise ValueError(f"{model_name}: 非膨脹樣本數過少，無法估計 normal component。")

    y = normal_dat[y_col].astype(float).values

    X_df = sm.add_constant(
        normal_dat[x_vars],
        has_constant="add"
    ).astype(float)

    if len(v_vars) > 0:
        Z_df = sm.add_constant(
            normal_dat[v_vars],
            has_constant="add"
        ).astype(float)
    else:
        Z_df = pd.DataFrame({"const": 1.0}, index=normal_dat.index)

    X = X_df.values
    Z = Z_df.values

    p = X.shape[1]
    k = Z.shape[1]

    beta_start = np.linalg.lstsq(X, y, rcond=None)[0]
    resid_start = y - X @ beta_start

    gamma_start = np.zeros(k)
    gamma_start[0] = np.log(np.var(resid_start) + 1e-6)

    theta_start = np.r_[beta_start, gamma_start]

    def nll(theta):
        beta = theta[:p]
        gamma = theta[p:]

        mu = X @ beta
        eta = np.clip(Z @ gamma, -20, 20)

        r = y - mu

        return 0.5 * np.sum(
            np.log(2 * np.pi)
            + eta
            + (r * r) * np.exp(-eta)
        )

    def grad(theta):
        beta = theta[:p]
        gamma = theta[p:]

        mu = X @ beta
        eta = np.clip(Z @ gamma, -20, 20)

        r = y - mu
        inv_var = np.exp(-eta)

        g_beta = -X.T @ (r * inv_var)
        g_gamma = 0.5 * Z.T @ (1 - (r * r) * inv_var)

        return np.r_[g_beta, g_gamma]

    opt = minimize(
        nll,
        theta_start,
        jac=grad,
        method="L-BFGS-B",
        options={
            "maxiter": 5000,
            "gtol": 1e-8,
            "ftol": 1e-12,
            "maxls": 50,
        },
    )

    H = approx_hess(opt.x, nll)
    cov = np.linalg.pinv(H)

    diag_cov = np.diag(cov)
    se = np.sqrt(np.where(diag_cov > 0, diag_cov, np.nan))

    beta_hat = opt.x[:p]
    gamma_hat = opt.x[p:]

    mu_hat_normal = X @ beta_hat
    eta_hat_normal = np.clip(Z @ gamma_hat, -20, 20)
    omega2_hat_normal = np.exp(eta_hat_normal)

    normal_resid = y - mu_hat_normal

    normal_loglik = float(-opt.fun)
    normal_mae = mae(y, mu_hat_normal)
    normal_rmse = rmse(y, mu_hat_normal)

    # Predict mu for all observations, not only non-inflated.
    X_all_df = sm.add_constant(
        dat[x_vars],
        has_constant="add"
    ).astype(float)

    if len(v_vars) > 0:
        Z_all_df = sm.add_constant(
            dat[v_vars],
            has_constant="add"
        ).astype(float)
    else:
        Z_all_df = pd.DataFrame({"const": 1.0}, index=dat.index)

    mu_hat_all = X_all_df.values @ beta_hat
    eta_hat_all = np.clip(Z_all_df.values @ gamma_hat, -20, 20)
    omega2_hat_all = np.exp(eta_hat_all)

    # Mixture mean prediction for all observations:
    # E[Y_it] = q_it * c + (1 - q_it) * mu_it
    mixture_mean_hat_all = (
        q_hat.values * point_mass_c
        + (1 - q_hat.values) * mu_hat_all
    )

    all_y = dat[y_col].astype(float).values
    mixture_mean_mae_all = mae(all_y, mixture_mean_hat_all)
    mixture_mean_rmse_all = rmse(all_y, mixture_mean_hat_all)

    total_loglik = float(q_loglik + normal_loglik)

    n_total = int(dat.shape[0])
    n_noninflated = int(normal_dat.shape[0])

    q_param_count = 2
    normal_param_count = int(p + k)
    total_param_count = int(q_param_count + normal_param_count)

    total_aic = float(2 * total_param_count - 2 * total_loglik)
    total_bic = float(np.log(n_total) * total_param_count - 2 * total_loglik)

    normal_aic = float(2 * normal_param_count - 2 * normal_loglik)
    normal_bic = float(np.log(n_noninflated) * normal_param_count - 2 * normal_loglik)

    # Coefficient summary
    names = []
    blocks = []
    components = []
    equation_parts = []

    for name in X_df.columns:
        if name == "const":
            blocks.append("beta_0")
            names.append("(Intercept)")
        elif name == F_COL:
            blocks.append("lambda")
            names.append(F_COL)
        else:
            blocks.append("beta_x")
            names.append(name)

        components.append("normal mean")
        equation_parts.append("mu_it = beta_0 + X beta_x" if F_COL not in x_vars else "mu_it = beta_0 + lambda F_it + X beta_x")

    for name in Z_df.columns:
        if name == "const":
            blocks.append("alpha_sigma")
            names.append("(Intercept)")
        else:
            blocks.append("beta_sigma")
            names.append(name)

        components.append("log variance")
        equation_parts.append("log(omega_it^2) = alpha_sigma + V beta_sigma")

    coef_df = pd.DataFrame({
        "model": model_name,
        "component": components,
        "parameter": blocks,
        "equation_part": equation_parts,
        "variable": names,
        "coef": opt.x,
        "se": se,
    })

    coef_df["z"] = coef_df["coef"] / coef_df["se"]
    coef_df["p_value"] = 2 * (1 - norm.cdf(np.abs(coef_df["z"])))
    coef_df["sig"] = coef_df["p_value"].apply(sig_star)
    coef_df["usable_for_significance"] = "Yes"
    coef_df["note"] = "non-inflated subset heteroscedastic normal MLE"

    # Fitted table
    # 重要：index 要保留 dat.index，否則後面用 normal_dat.index 回填殘差時會 KeyError。
    fitted_df = pd.DataFrame(
        {
            "model": model_name,
            "row_index": dat.index,
            "y": dat[y_col].astype(float).values,
            "true_inflated": dat["true_inflated"].astype(int).values,
            "q_hat": q_hat.loc[dat.index].values if isinstance(q_hat, pd.Series) else np.asarray(q_hat),
            "mu_hat": mu_hat_all,
            "log_omega2_hat": eta_hat_all,
            "omega2_hat": omega2_hat_all,
            "omega_hat": np.sqrt(omega2_hat_all),
            "mixture_mean_hat": mixture_mean_hat_all,
            "mixture_mean_resid": all_y - mixture_mean_hat_all,
        },
        index=dat.index
    )

    # Normal-only residual is meaningful only for non-inflated component.
    fitted_df["normal_component_resid"] = np.nan
    fitted_df.loc[normal_dat.index, "normal_component_resid"] = normal_resid

    model_metrics = {
        "model": model_name,
        "normal_mean_structure": (
            "mu_it = beta_0 + X_it beta_x"
            if F_COL not in x_vars
            else "mu_it = beta_0 + X_it beta_x + lambda F_it"
        ),
        "variance_structure": "log(omega_it^2) = alpha_sigma + V_it beta_sigma",
        "n_total": n_total,
        "n_inflated": int(dat["true_inflated"].sum()),
        "n_noninflated": n_noninflated,
        "n_mean_variables_without_intercept": int(len(x_vars)),
        "n_variance_variables_without_intercept": int(len(v_vars)),
        "normal_loglik": normal_loglik,
        "q_loglik": float(q_loglik),
        "total_mixture_loglik": total_loglik,
        "normal_component_MAE": normal_mae,
        "normal_component_RMSE": normal_rmse,
        "mixture_mean_MAE_all": mixture_mean_mae_all,
        "mixture_mean_RMSE_all": mixture_mean_rmse_all,
        "normal_component_AIC": normal_aic,
        "normal_component_BIC": normal_bic,
        "total_mixture_AIC": total_aic,
        "total_mixture_BIC": total_bic,
        "q_params": q_param_count,
        "normal_params": normal_param_count,
        "total_params": total_param_count,
        "optimizer_success": str(opt.success),
        "optimizer_message": str(opt.message),
    }

    return coef_df, fitted_df, model_metrics


coef_x, fitted_x, metrics_x = fit_normal_component(
    dat=dat,
    model_name="MIX-X: specified covariates only",
    x_vars=X_ONLY_VARS,
    v_vars=V_vars,
    y_col=Y_COL,
    q_hat=q_hat,
    q_loglik=q_loglik,
    point_mass_c=POINT_MASS_C,
)

coef_xf, fitted_xf, metrics_xf = fit_normal_component(
    dat=dat,
    model_name="MIX-XF: specified covariates + F_it",
    x_vars=X_F_VARS,
    v_vars=V_vars,
    y_col=Y_COL,
    q_hat=q_hat,
    q_loglik=q_loglik,
    point_mass_c=POINT_MASS_C,
)


# =========================================================
# 13) Model comparison
# =========================================================
model_comparison = pd.DataFrame([metrics_x, metrics_xf])

# Add improvement row: MIX-X minus MIX-XF
improvement = {
    "model": "Improvement: MIX-X minus MIX-XF",
    "normal_mean_structure": "positive values mean MIX-XF is better for error/AIC/BIC",
    "variance_structure": "same as above",
    "n_total": np.nan,
    "n_inflated": np.nan,
    "n_noninflated": np.nan,
    "n_mean_variables_without_intercept": np.nan,
    "n_variance_variables_without_intercept": np.nan,
    "normal_loglik": metrics_xf["normal_loglik"] - metrics_x["normal_loglik"],
    "q_loglik": 0.0,
    "total_mixture_loglik": metrics_xf["total_mixture_loglik"] - metrics_x["total_mixture_loglik"],
    "normal_component_MAE": metrics_x["normal_component_MAE"] - metrics_xf["normal_component_MAE"],
    "normal_component_RMSE": metrics_x["normal_component_RMSE"] - metrics_xf["normal_component_RMSE"],
    "mixture_mean_MAE_all": metrics_x["mixture_mean_MAE_all"] - metrics_xf["mixture_mean_MAE_all"],
    "mixture_mean_RMSE_all": metrics_x["mixture_mean_RMSE_all"] - metrics_xf["mixture_mean_RMSE_all"],
    "normal_component_AIC": metrics_x["normal_component_AIC"] - metrics_xf["normal_component_AIC"],
    "normal_component_BIC": metrics_x["normal_component_BIC"] - metrics_xf["normal_component_BIC"],
    "total_mixture_AIC": metrics_x["total_mixture_AIC"] - metrics_xf["total_mixture_AIC"],
    "total_mixture_BIC": metrics_x["total_mixture_BIC"] - metrics_xf["total_mixture_BIC"],
    "q_params": np.nan,
    "normal_params": np.nan,
    "total_params": np.nan,
    "optimizer_success": "",
    "optimizer_message": "",
}

model_comparison = pd.concat(
    [model_comparison, pd.DataFrame([improvement])],
    ignore_index=True
)


# =========================================================
# 14) Coefficient tables
# =========================================================
coef_table = pd.concat(
    [
        q_rows,
        coef_x,
        coef_xf,
    ],
    ignore_index=True
)

significant_only = coef_table[
    (coef_table["sig"].isin(["***", "**", "*", "."])) |
    (coef_table["parameter"].isin(["alpha_p", "beta_p", "lambda"]))
].copy()


# =========================================================
# 15) Model information
# =========================================================
model_info_records = [
    ["source_data_file", str(DATA_FILE)],
    ["source_sheet", RESULT_SHEET],
    ["outcome_column", Y_COL],
    ["point_mass_c", POINT_MASS_C],
    ["n_rows_original", int(n_rows_original)],
    ["n_rows_after_mean_imputation", int(len(df))],
    ["n_rows_entering_mixture_model", int(dat.shape[0])],
    ["n_rows_removed_after_imputation", int(n_rows_original - dat.shape[0])],
    ["n_total_cells_imputed_by_mean", int(imputation_table["n_missing_before"].sum())],
    ["n_imputed_columns", int((imputation_table["n_missing_before"] > 0).sum())],
    ["n_inflated", int(dat["true_inflated"].sum())],
    ["n_noninflated", int((dat["true_inflated"] == 0).sum())],
    ["n_continuous_z_variables", int(len(continuous_z_cols))],
    ["n_proportion_logit_z_variables", int(len(proportion_logit_z_cols))],
    ["n_MIX_X_mean_variables", int(len(X_ONLY_VARS))],
    ["n_MIX_XF_mean_variables", int(len(X_F_VARS))],
    ["n_variance_variables", int(len(V_vars))],
]

for k, v in q_info.items():
    model_info_records.append([k, v])

model_info = pd.DataFrame(model_info_records, columns=["item", "value"])


# =========================================================
# 16) Variable design tables
# =========================================================
variable_design_records = []

for model_name, x_vars in [
    ("MIX-X: specified covariates only", X_ONLY_VARS),
    ("MIX-XF: specified covariates + F_it", X_F_VARS),
]:
    for c in x_vars:
        if c == F_COL:
            raw_name = F_COL
            role = "normal_mean_BayesNF_F"
            transformation = "none"
            standardized = False
        elif c in continuous_z_cols:
            raw_name = c.replace("_z", "")
            role = "normal_mean_continuous_X"
            transformation = "x -> z-score"
            standardized = True
        elif c in proportion_logit_z_cols:
            raw_name = c.replace("_logit_z", "")
            role = "normal_mean_proportion_X"
            transformation = "p -> logit(p) -> z-score"
            standardized = True
        else:
            raw_name = c
            role = "normal_mean_X"
            transformation = "unknown"
            standardized = False

        variable_design_records.append({
            "model": model_name,
            "equation": "normal_mean",
            "variable_used_in_model": c,
            "raw_variable": raw_name,
            "role": role,
            "transformation": transformation,
            "standardized": standardized,
        })

for c in V_vars:
    if c in continuous_z_cols:
        raw_name = c.replace("_z", "")
        transformation = "x -> z-score"
        standardized = True
    elif c in proportion_logit_z_cols:
        raw_name = c.replace("_logit_z", "")
        transformation = "p -> logit(p) -> z-score"
        standardized = True
    elif c in ["high_risk", "potential_high_risk"]:
        raw_name = c
        transformation = "raw 0/1 indicator"
        standardized = False
    else:
        raw_name = c
        transformation = "unknown"
        standardized = False

    variable_design_records.append({
        "model": "both MIX-X and MIX-XF",
        "equation": "log_variance",
        "variable_used_in_model": c,
        "raw_variable": raw_name,
        "role": "log_variance_V",
        "transformation": transformation,
        "standardized": standardized,
    })

variable_design_table = pd.DataFrame(variable_design_records)

reference_table = pd.DataFrame({
    "removed_reference_variable": reference_cols,
    "reason": "reference group to avoid perfect multicollinearity"
})

continuous_transform_table = transform_map[
    transform_map["variable_type"] == "specified_continuous"
].copy()

proportion_transform_table = transform_map[
    transform_map["variable_type"] == "non_specified_continuous_treated_as_proportion"
].copy()

fitted_values = pd.concat(
    [fitted_x, fitted_xf],
    ignore_index=True
)


# =========================================================
# 17) Export
# =========================================================
with pd.ExcelWriter(OUT_FILE, engine="openpyxl") as writer:
    model_comparison.to_excel(writer, sheet_name="Model_Comparison", index=False)
    coef_table.to_excel(writer, sheet_name="Coefficient_Table", index=False)
    significant_only.to_excel(writer, sheet_name="Significant_Only", index=False)
    model_info.to_excel(writer, sheet_name="Model_Info", index=False)
    imputation_table.to_excel(writer, sheet_name="Mean_Imputation", index=False)
    transform_map.to_excel(writer, sheet_name="Transform_Map", index=False)
    continuous_transform_table.to_excel(writer, sheet_name="Continuous_Z_Params", index=False)
    proportion_transform_table.to_excel(writer, sheet_name="Proportion_Logit_Z", index=False)
    variable_design_table.to_excel(writer, sheet_name="Variable_Design", index=False)
    reference_table.to_excel(writer, sheet_name="Reference_Cols", index=False)
    dat.to_excel(writer, sheet_name="Model_Used_Data", index=False)
    fitted_values.to_excel(writer, sheet_name="Fitted_Values", index=False)


# =========================================================
# 18) Print results
# =========================================================
print("\n==============================")
print("Mean imputation summary")
print("==============================")
print(imputation_table[imputation_table["n_missing_before"] > 0])

print("\n==============================")
print("Model comparison")
print("==============================")
print(model_comparison)

print("\n==============================")
print("Coefficient table")
print("==============================")
print(coef_table)

print("\n==============================")
print("Significant only")
print("==============================")
print(significant_only)

print("\n==============================")
print("Mean equation variables: MIX-X")
print("==============================")
print(X_ONLY_VARS)

print("\n==============================")
print("Mean equation variables: MIX-XF")
print("==============================")
print(X_F_VARS)

print("\n==============================")
print("Variance equation variables")
print("==============================")
print(V_vars)

print("\n==============================")
print("Continuous z variables")
print("==============================")
print(continuous_z_cols)

print("\n==============================")
print("Proportion logit-z variables")
print("==============================")
print(proportion_logit_z_cols)

print(f"\nSaved: {OUT_FILE}")

資料筆數： 1297
資料欄位數： 90
平均補值欄位數： 37
平均補值總格數： 7
補值後總資料筆數： 1297
轉換前資料筆數（補值後，未刪列）： 1297
膨脹點筆數： 424
非膨脹點筆數： 873
指定連續型 z-score 變數數量： 13
比例變數 logit-z 變數數量： 20
進入 mixture model 的資料筆數（未刪列）： 1297
進入 mixture model 的膨脹點筆數： 424
進入 mixture model 的非膨脹點筆數： 873
MIX-X mean variables 數量： 33
MIX-XF mean variables 數量： 34
Variance equation V_vars 數量： 9

Mean imputation summary
          variable  n_missing_before  mean_used_for_imputation  \
4       Lead_water                 5                  0.003198   
11  平均BMI  (kg/m2)                 2                 23.657402   

    n_missing_after                  status  
4                 0  imputed_by_column_mean  
11                0  imputed_by_column_mean  

Model comparison
                                 model  \
0     MIX-X: specified covariates only   
1  MIX-XF: specified covariates + F_it   
2      Improvement: MIX-X minus MIX-XF   

                               normal_mean_structure  \
0                       mu_it = beta_0 + X_it beta_x   
1       

In [1]:
# -*- coding: utf-8 -*-
"""
Fit the inflated-normal calibration framework:

Y_it ~ q_it delta_c + (1-q_it) N(mu_it, omega_it^2)
q_it = logit^{-1}[alpha_p + beta_p logit(p_BNF_it)]
mu_it = beta_0 + lambda F_BNF_it + X_it^T beta_x
log(omega_it^2) = alpha_sigma + V_it^T beta_sigma

Outputs:
1) coefficient table with alpha_p, beta_p, beta_0, lambda, beta_x, alpha_sigma, beta_sigma
2) significance stars
3) model diagnostics

Note:
For a point-mass + continuous normal mixture, if y == c the likelihood contribution is q_it,
and if y != c the contribution is (1-q_it) * Normal(y; mu_it, omega_it^2).
Therefore the model can be fit as:
- q-model: logistic regression for inflated indicator.
- non-inflated model: heteroscedastic normal likelihood for y != c.
"""

import os
import re
import warnings
import numpy as np
import pandas as pd
import statsmodels.api as sm
from scipy.optimize import minimize
from scipy.special import logit
from scipy.stats import norm
from statsmodels.tools.numdiff import approx_hess
from sklearn.metrics import roc_auc_score

# =========================
# 0) File paths
# =========================
RESULT_FILE = "C:/Users/User/Desktop/inflated_normal_full_individualized_sigma_results.xlsx"
RAW_FILE    = "C:/Users/User/Desktop/LC analysis/gdf_table_符合水痘_死亡資料變換可數型.xlsx"
OUT_FILE    = "C:/Users/User/Desktop/inflated_normal_mixture_coefficients.xlsx"

RESULT_SHEET = "row_level"

# =========================
# 1) Basic utilities
# =========================
def zscore(s):
    s = pd.to_numeric(s, errors="coerce")
    sd = s.std(ddof=0)
    if pd.isna(sd) or sd == 0:
        return pd.Series(np.nan, index=s.index)
    return (s - s.mean()) / sd

def safe_logit01(s, eps=1e-4):
    s = pd.to_numeric(s, errors="coerce")
    s = np.clip(s, eps, 1 - eps)
    return np.log(s / (1 - s))

def make_logit_z(df, col, eps=1e-4):
    return zscore(safe_logit01(df[col], eps=eps))

def unique(seq):
    out, seen = [], set()
    for x in seq:
        if x not in seen:
            out.append(x)
            seen.add(x)
    return out

def sig_star(p):
    if pd.isna(p):
        return ""
    if p < 0.001:
        return "***"
    if p < 0.01:
        return "**"
    if p < 0.05:
        return "*"
    if p < 0.10:
        return "."
    return ""

# =========================
# 2) Read and merge data
# =========================
row = pd.read_excel(RESULT_FILE, sheet_name=RESULT_SHEET)
raw = pd.read_excel(RAW_FILE, sheet_name=0)

for d in (row, raw):
    d["年份"] = pd.to_datetime(d["年份"], errors="coerce")
    d["鄉鎮"] = d["鄉鎮"].astype(str).str.strip().str.replace("臺", "台", regex=False)

# Rename Excel's duplicated headers to meaningful names.
raw = raw.rename(columns={
    "癌症期別": "癌症期別1",
    "...17": "癌症期別2",
    "癌症組織類型": "癌症組織類型1",
    "...21": "癌症組織類型2",
    "...22": "癌症組織類型3",
    "...23": "癌症組織類型4",
})

df = row.merge(
    raw.drop(columns=[c for c in ["死亡_常態"] if c in raw.columns]),
    on=["年份", "鄉鎮"],
    how="left",
    suffixes=("", "_raw")
)

# If true_inflated is unavailable, infer from y_imputed == point mass c.
if "true_inflated" not in df.columns:
    c_point = -6.907255070523717
    df["true_inflated"] = (np.abs(df["y_imputed"] - c_point) < 1e-8).astype(int)

# =========================
# 3) Transform covariates
# =========================
tf = df.copy()

# BNF inflation probability calibration covariate
tf["logit_p_BNF"] = safe_logit01(tf["p_mean"], eps=1e-6)

# Continuous/count variables: z-score
continuous_vars = [
    "該鄉鎮總人口數 (人數)",
    "小於18歲人口數 (人數)",
    "Lead_water",
    "PM2.5 (ug/m3)",
    "Income_regular (百萬元)",
    "每百萬人病床數 (床/萬人)",
    "平均年齡（歲）",
    "平均BMI  (kg/m2)",
    "平均日常體能狀態（分）",
    "加權 CCI",
    "平均診斷到index date時間(日)",
    "平均健保投保金額",
]

# Proportion variables: clipped logit then z-score.
# For compositional blocks, drop one reference category to avoid collinearity:
# - stage: keep 癌症期別1, drop 癌症期別2
# - histology: keep 1,2,3, drop 4
ratio_vars = [
    "性別(男)",
    "癌症期別1",
    "仍吸菸",
    "仍喝酒",
    "癌症組織類型1",
    "癌症組織類型2",
    "癌症組織類型3",
    "OP指標日期前",
    "RT指標日期後",
    "併用藥物1",
    "併用藥物2",
    "併用藥物3",
    "併用藥物4",
    "併用藥物5",
    "併用藥物6",
    "併用藥物7",
    "併用藥物8",
    "併用藥物9",
    "併用藥物10",
    "併用藥物11",
    "醫學中心",
]

transformed_cols = {}

for col in continuous_vars:
    if col in tf.columns:
        new = f"{col}_z"
        tf[new] = zscore(tf[col])
        transformed_cols[col] = new

# Lead_soil: scale by maximum as upper bound, then logit-z.
if "Lead_soil" in tf.columns:
    max_soil = pd.to_numeric(tf["Lead_soil"], errors="coerce").max()
    tf["Lead_soil_ratio_logit_z"] = zscore(
        safe_logit01(pd.to_numeric(tf["Lead_soil"], errors="coerce") / max_soil, eps=1e-4)
    )
    transformed_cols["Lead_soil"] = "Lead_soil_ratio_logit_z"

for col in ratio_vars:
    if col in tf.columns:
        new = f"{col}_logit_z"
        tf[new] = make_logit_z(tf, col)
        transformed_cols[col] = new

# =========================
# 4) Choose X and V
# =========================
# Mean equation:
X_vars = (
    ["F_mean"]
    + [transformed_cols[c] for c in continuous_vars if c in transformed_cols]
    + ([transformed_cols["Lead_soil"]] if "Lead_soil" in transformed_cols else [])
    + [transformed_cols[c] for c in ratio_vars if c in transformed_cols]
)

# Variance equation:
# Keep this smaller than X by default for numerical stability.
# You can edit this list if you want variance to depend on more variables.
V_vars = [
    "Lead_water_z",
    "Lead_soil_ratio_logit_z",
    "PM2.5 (ug/m3)_z",
    "每百萬人病床數 (床/萬人)_z",
    "平均年齡（歲）_z",
    "平均日常體能狀態（分）_z",
    "加權 CCI_z",
    "平均診斷到index date時間(日)_z",
    "high_risk",
    "potential_high_risk",
]
V_vars = [v for v in V_vars if v in tf.columns]

needed = unique(["y_imputed", "true_inflated", "p_mean", "logit_p_BNF"] + X_vars + V_vars)
dat = tf[needed].replace([np.inf, -np.inf], np.nan).dropna().copy()
dat["true_inflated"] = dat["true_inflated"].astype(int)

# =========================
# 5) q-model: alpha_p, beta_p
# =========================
y_bin = dat["true_inflated"].values
Xq = sm.add_constant(dat[["logit_p_BNF"]], has_constant="add")

q_note = ""# -*- coding: utf-8 -*-
"""
Inflated-normal mixture regression with two normal-mean designs.

Mixture observation model:
Y_it ~ q_it delta_c + (1 - q_it) N(mu_it, omega_it^2)

Common inflation probability model:
q_it = logit^{-1}[alpha_p + beta_p logit(p_BNF_it)]

Normal component model 1:
MIX-X:
mu_it = beta_0 + X_it^T beta_x

Normal component model 2:
MIX-XF:
mu_it = beta_0 + X_it^T beta_x + lambda F_it

Log variance model:
log(omega_it^2) = alpha_sigma + V_it^T beta_sigma

Variable transformation:
1. Specified continuous variables:
   x -> z-score

2. Other non-specified continuous variables:
   treated as proportion variables
   p -> logit(p) -> z-score

Output:
1. Coefficient table for q-model and both normal components
2. Model comparison table: MAE, RMSE, logLik, AIC, BIC
3. Transformation map
4. Model-used data with fitted values
"""

import warnings
import numpy as np
import pandas as pd
import statsmodels.api as sm

from pathlib import Path
from scipy.optimize import minimize
from scipy.stats import norm
from statsmodels.tools.numdiff import approx_hess
from statsmodels.tools.sm_exceptions import PerfectSeparationError
from sklearn.metrics import roc_auc_score


# =========================================================
# 0) File paths
# =========================================================
DATA_FILE = Path(
    r"C:/Users/User/Desktop/LC analysis/result/inflated_normal_highrisk_five_features_results.xlsx"
)

OUT_FILE = Path(
    r"C:/Users/User/Desktop/LC analysis/result/inflated_normal_mixture_X_vs_XF_comparison.xlsx"
)

RESULT_SHEET = "row_level"


# =========================================================
# 1) Basic utilities
# =========================================================
def unique(seq):
    out, seen = [], set()
    for x in seq:
        if x not in seen:
            out.append(x)
            seen.add(x)
    return out


def sig_star(p):
    if pd.isna(p):
        return ""
    if p < 0.001:
        return "***"
    if p < 0.01:
        return "**"
    if p < 0.05:
        return "*"
    if p < 0.10:
        return "."
    return ""


def parse_year(s):
    s = pd.Series(s).copy()
    numeric = pd.to_numeric(s, errors="coerce")

    out = pd.Series(index=s.index, dtype="float")

    mask_excel_date = numeric.between(30000, 60000)
    out.loc[mask_excel_date] = pd.to_datetime(
        numeric.loc[mask_excel_date],
        unit="D",
        origin="1899-12-30",
        errors="coerce"
    ).dt.year

    mask_year = numeric.between(1900, 2100)
    out.loc[mask_year] = numeric.loc[mask_year]

    mask_rest = out.isna()
    out.loc[mask_rest] = pd.to_datetime(
        s.loc[mask_rest],
        errors="coerce"
    ).dt.year

    return out.astype("Int64")


def clean_town(x):
    return (
        x.astype(str)
        .str.strip()
        .str.replace("臺", "台", regex=False)
        .str.replace(" ", "", regex=False)
    )


def zscore_series(s):
    s = pd.to_numeric(s, errors="coerce")
    mean_s = s.mean()
    sd_s = s.std(ddof=0)

    if pd.isna(sd_s) or sd_s <= 0:
        return None, mean_s, sd_s

    return (s - mean_s) / sd_s, mean_s, sd_s


def safe_logit01(s, eps=1e-6):
    s = pd.to_numeric(s, errors="coerce")
    s_clip = np.clip(s, eps, 1 - eps)
    return np.log(s_clip / (1 - s_clip))


def proportion_to_01(s, var_name):
    """
    將比例變數轉成 [0,1] 尺度。

    可接受：
    1. 原本就是比例，例如 0.32
    2. 百分比格式，例如 32.0，會自動除以 100

    若數值不在 [0,1] 或 [0,100]，會報錯提醒。
    """
    s = pd.to_numeric(s, errors="coerce")

    min_s = s.min(skipna=True)
    max_s = s.max(skipna=True)

    if pd.isna(min_s) or pd.isna(max_s):
        raise ValueError(f"{var_name} 無法判斷比例範圍，可能全為缺失值。")

    if min_s >= 0 and max_s <= 1:
        p = s.copy()
        scale_type = "0_to_1"

    elif min_s >= 0 and max_s <= 100:
        p = s / 100
        scale_type = "0_to_100_divided_by_100"

    else:
        raise ValueError(
            f"{var_name} 被設定為比例變數，但數值範圍為 "
            f"[{min_s}, {max_s}]，不在 [0,1] 或 [0,100] 內。"
        )

    return p, scale_type


def safe_logit_with_clip_count(p, eps=1e-6):
    p = pd.to_numeric(p, errors="coerce")

    n_lower_clip = int((p <= eps).sum())
    n_upper_clip = int((p >= 1 - eps).sum())

    p_clip = p.clip(lower=eps, upper=1 - eps)
    logit_value = np.log(p_clip / (1 - p_clip))

    return logit_value, p_clip, n_lower_clip, n_upper_clip


def rmse(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    return float(np.sqrt(np.mean((y_true - y_pred) ** 2)))


def mae(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)
    return float(np.mean(np.abs(y_true - y_pred)))


# =========================================================
# 2) Read data
# =========================================================
df = pd.read_excel(DATA_FILE, sheet_name=RESULT_SHEET)

print("資料筆數：", len(df))
print("資料欄位數：", df.shape[1])

# 若 Excel 讀進來有重複欄位名稱，可補正。
df = df.rename(columns={
    "癌症期別": "癌症期別1",
    "...17": "癌症期別2",
    "癌症組織類型": "癌症組織類型1",
    "...21": "癌症組織類型2",
    "...22": "癌症組織類型3",
    "...23": "癌症組織類型4",
})

df = df.copy()

if "年份" in df.columns:
    df["year"] = parse_year(df["年份"])

if "鄉鎮" in df.columns:
    df["鄉鎮_key"] = clean_town(df["鄉鎮"])
    df["鄉鎮6"] = df["鄉鎮_key"].str[:6]


# =========================================================
# 3) Define outcome, point mass, and BayesNF outputs
# =========================================================
POINT_MASS_C = -6.907255070523717

if "死亡_cll" in df.columns:
    Y_COL = "死亡_cll"
elif "y_imputed" in df.columns:
    Y_COL = "y_imputed"
elif "y_true" in df.columns:
    Y_COL = "y_true"
else:
    raise ValueError("資料缺少反應變數欄位：死亡_cll、y_imputed、y_true 都不存在。")

F_COL = "F_mean"
P_COL = "p_mean"

for c in [Y_COL, F_COL, P_COL]:
    if c not in df.columns:
        raise ValueError(f"資料缺少必要欄位：{c}")

df[Y_COL] = pd.to_numeric(df[Y_COL], errors="coerce")
df[F_COL] = pd.to_numeric(df[F_COL], errors="coerce")
df[P_COL] = pd.to_numeric(df[P_COL], errors="coerce")

if "true_inflated" not in df.columns:
    df["true_inflated"] = (
        np.abs(pd.to_numeric(df[Y_COL], errors="coerce") - POINT_MASS_C) < 1e-8
    ).astype(int)
else:
    df["true_inflated"] = pd.to_numeric(df["true_inflated"], errors="coerce").astype("Int64")


# =========================================================
# 4) Covariates
# =========================================================
covariates = [
    "該鄉鎮總人口數 (人數)",
    # "小於18歲人口數 (人數)",
    "Lead_water",
    "Lead_soil",
    "PM2.5 (ug/m3)",
    "Income_regular (百萬元)",
    "每百萬人病床數 (床/萬人)",
    "平均年齡（歲）",
    "性別(男)",
    "平均BMI  (kg/m2)",
    "平均日常體能狀態（分）",
    "加權 CCI",
    "癌症期別1",
    "癌症期別2",
    "仍吸菸",
    "仍喝酒",
    "癌症組織類型1",
    "癌症組織類型2",
    "癌症組織類型3",
    "癌症組織類型4",
    "OP指標日期前",
    "RT指標日期後",
    "併用藥物1",
    "併用藥物2",
    "併用藥物3",
    "併用藥物4",
    "併用藥物5",
    "併用藥物6",
    "併用藥物7",
    "併用藥物8",
    "併用藥物9",
    "併用藥物10",
    "併用藥物11",
    "平均診斷到index date時間(日)",
    "平均健保投保金額",
    "醫學中心",
]

covariates = [c for c in covariates if c in df.columns]

# 移除 reference category，避免完全共線性。
reference_cols = [
    "癌症期別2",
    "癌症組織類型4",
]
reference_cols = [c for c in reference_cols if c in covariates]

x_raw_cols = [c for c in covariates if c not in reference_cols]

# 避免 outcome 或 BayesNF 輸出被放進一般 X
leakage_cols = [
    "死亡",
    "死亡_cll",
    "死亡_logit",
    "死亡_probit",
    "death_count",
    "death_count_raw",
    "y_true",
    "y_imputed",
    "F_mean",
    "p_mean",
    "sigma_mean",
    "sigma2_mean",
    "F_q05",
    "F_q95",
    "p_q05",
    "p_q95",
    "sigma_q05",
    "sigma_q95",
    "sigma2_q05",
    "sigma2_q95",
    "yhat_mix_mean",
    "yhat_mix_q05",
    "yhat_mix_q95",
    "err_mix",
    "abs_err_mix",
    "sq_err_mix",
    "err_F_only",
    "abs_err_F_only",
    "sq_err_F_only",
    "pred_inflated_05",
    "true_inflated",
]

x_raw_cols = [c for c in x_raw_cols if c not in leakage_cols]

for c in x_raw_cols:
    df[c] = pd.to_numeric(df[c], errors="coerce")

# 移除沒有變異的 X
x_raw_cols = [
    c for c in x_raw_cols
    if c in df.columns and df[c].nunique(dropna=True) > 1
]


# =========================================================
# 5) Variable type design
#
# Specified continuous variables:
# x -> z-score
#
# Other variables:
# p -> logit(p) -> z-score
# =========================================================
CONTINUOUS_COLS = [
    "該鄉鎮總人口數 (人數)",
    "Lead_water",
    "Lead_soil",
    "PM2.5 (ug/m3)",
    "Income_regular (百萬元)",
    "每百萬人病床數 (床/萬人)",
    "平均年齡（歲）",
    "性別(男)",
    "平均BMI  (kg/m2)",
    "平均日常體能狀態（分）",
    "加權 CCI",
    "平均診斷到index date時間(日)",
    "平均健保投保金額",
]

continuous_cols = [
    c for c in CONTINUOUS_COLS
    if c in x_raw_cols and c in df.columns
]

proportion_cols = [
    c for c in x_raw_cols
    if c not in continuous_cols
]


# =========================================================
# 6) Build base data
# =========================================================
id_cols = [
    c for c in ["年份", "year", "鄉鎮", "鄉鎮_key", "鄉鎮6"]
    if c in df.columns
]

risk_cols = [
    c for c in ["high_risk", "potential_high_risk"]
    if c in df.columns
]

needed_raw = unique(
    id_cols
    + [Y_COL, "true_inflated", F_COL, P_COL]
    + x_raw_cols
    + risk_cols
)

tf = (
    df[needed_raw]
    .replace([np.inf, -np.inf], np.nan)
    .copy()
)

tf = tf.dropna(subset=[Y_COL, "true_inflated", F_COL, P_COL] + x_raw_cols).copy()
tf["true_inflated"] = tf["true_inflated"].astype(int)

print("轉換前可用資料筆數：", len(tf))
print("膨脹點筆數：", int(tf["true_inflated"].sum()))
print("非膨脹點筆數：", int((tf["true_inflated"] == 0).sum()))


# =========================================================
# 7) Transform BNF p_mean
# =========================================================
tf["logit_p_BNF"] = safe_logit01(tf[P_COL], eps=1e-6)


# =========================================================
# 8) Transform covariates
# =========================================================
transform_records = []
transformed_cols = {}

continuous_z_cols = []
proportion_logit_z_cols = []

# A) Specified continuous: x -> z-score
for c in continuous_cols:
    z_col = f"{c}_z"

    z_value, mean_c, sd_c = zscore_series(tf[c])

    if z_value is not None:
        tf[z_col] = z_value
        transformed_cols[c] = z_col
        continuous_z_cols.append(z_col)

        transform_records.append({
            "raw_variable": c,
            "model_variable": z_col,
            "variable_type": "specified_continuous",
            "transformation": "x -> z-score",
            "proportion_scale_type": "",
            "mean_used_for_z": mean_c,
            "sd_used_for_z": sd_c,
            "epsilon_for_logit": np.nan,
            "n_lower_clipped": np.nan,
            "n_upper_clipped": np.nan,
        })
    else:
        print(f"連續變數標準化略過：{c}，因為標準差為 0 或無法計算。")


# B) Other variables: p -> logit(p) -> z-score
for c in proportion_cols:
    prop_col = f"{c}_prop01"
    logit_col = f"{c}_logit"
    logit_z_col = f"{c}_logit_z"

    p_raw, scale_type = proportion_to_01(tf[c], c)
    logit_value, p_clip, n_lower_clip, n_upper_clip = safe_logit_with_clip_count(
        p_raw,
        eps=1e-6
    )

    tf[prop_col] = p_clip
    tf[logit_col] = logit_value

    z_value, mean_logit, sd_logit = zscore_series(tf[logit_col])

    if z_value is not None:
        tf[logit_z_col] = z_value
        transformed_cols[c] = logit_z_col
        proportion_logit_z_cols.append(logit_z_col)

        transform_records.append({
            "raw_variable": c,
            "model_variable": logit_z_col,
            "variable_type": "non_specified_continuous_treated_as_proportion",
            "transformation": "p -> logit(p) -> z-score",
            "proportion_scale_type": scale_type,
            "mean_used_for_z": mean_logit,
            "sd_used_for_z": sd_logit,
            "epsilon_for_logit": 1e-6,
            "n_lower_clipped": n_lower_clip,
            "n_upper_clipped": n_upper_clip,
        })
    else:
        print(f"比例變數 logit 後標準化略過：{c}，因為 logit 後標準差為 0 或無法計算。")


transform_map = pd.DataFrame(transform_records)

print("指定連續型 z-score 變數數量：", len(continuous_z_cols))
print("比例變數 logit-z 變數數量：", len(proportion_logit_z_cols))


# =========================================================
# 9) Define mean equation and variance equation variables
# =========================================================
# MIX-X: only specified transformed covariates
X_ONLY_VARS = unique(
    continuous_z_cols
    + proportion_logit_z_cols
)

# MIX-XF: specified transformed covariates + BayesNF F_it
X_F_VARS = unique(
    [F_COL]
    + continuous_z_cols
    + proportion_logit_z_cols
)

# Variance equation:
# keep same V structure for both models.
V_raw_candidates = [
    "Lead_water",
    "Lead_soil",
    "PM2.5 (ug/m3)",
    "每百萬人病床數 (床/萬人)",
    "平均年齡（歲）",
    "平均日常體能狀態（分）",
    "加權 CCI",
    "平均診斷到index date時間(日)",
]

V_vars = []
for raw_c in V_raw_candidates:
    if raw_c in transformed_cols:
        V_vars.append(transformed_cols[raw_c])

for risk_col in ["high_risk", "potential_high_risk"]:
    if risk_col in tf.columns:
        tf[risk_col] = pd.to_numeric(tf[risk_col], errors="coerce")
        if tf[risk_col].nunique(dropna=True) > 1:
            V_vars.append(risk_col)

V_vars = unique([v for v in V_vars if v in tf.columns])

needed_model = unique(
    [Y_COL, "true_inflated", P_COL, "logit_p_BNF"]
    + X_ONLY_VARS
    + X_F_VARS
    + V_vars
)

dat = (
    tf[needed_model]
    .replace([np.inf, -np.inf], np.nan)
    .dropna()
    .copy()
)

dat["true_inflated"] = dat["true_inflated"].astype(int)

print("進入 mixture model 的資料筆數：", len(dat))
print("進入 mixture model 的膨脹點筆數：", int(dat["true_inflated"].sum()))
print("進入 mixture model 的非膨脹點筆數：", int((dat["true_inflated"] == 0).sum()))
print("MIX-X mean variables 數量：", len(X_ONLY_VARS))
print("MIX-XF mean variables 數量：", len(X_F_VARS))
print("Variance equation V_vars 數量：", len(V_vars))


# =========================================================
# 10) q-model: alpha_p, beta_p
# =========================================================
def fit_q_model(dat, p_col=P_COL):
    y_bin = dat["true_inflated"].astype(int).values
    Xq = sm.add_constant(dat[["logit_p_BNF"]], has_constant="add")

    q_note = ""
    q_complete_separation = False
    q_usable_for_significance = True

    try:
        with warnings.catch_warnings(record=True) as w:
            warnings.simplefilter("always")
            q_res = sm.Logit(y_bin, Xq).fit(disp=False, maxiter=200)
            q_note = "; ".join(sorted(set([str(ww.message) for ww in w])))

        q_params = q_res.params
        q_bse = q_res.bse
        q_z = q_res.tvalues
        q_pvalues = q_res.pvalues

    except PerfectSeparationError as e:
        q_complete_separation = True
        q_usable_for_significance = False
        q_note = f"PerfectSeparationError: {str(e)}"

        glm_binom = sm.GLM(y_bin, Xq, family=sm.families.Binomial())
        q_res_reg = glm_binom.fit_regularized(alpha=1e-6, L1_wt=0.0, maxiter=1000)

        q_params = pd.Series(q_res_reg.params, index=Xq.columns)
        q_bse = pd.Series(np.nan, index=Xq.columns)
        q_z = pd.Series(np.nan, index=Xq.columns)
        q_pvalues = pd.Series(np.nan, index=Xq.columns)

    except Exception as e:
        q_complete_separation = True
        q_usable_for_significance = False
        q_note = f"Logit failed and fallback used: {str(e)}"

        glm_binom = sm.GLM(y_bin, Xq, family=sm.families.Binomial())
        q_res_reg = glm_binom.fit_regularized(alpha=1e-6, L1_wt=0.0, maxiter=1000)

        q_params = pd.Series(q_res_reg.params, index=Xq.columns)
        q_bse = pd.Series(np.nan, index=Xq.columns)
        q_z = pd.Series(np.nan, index=Xq.columns)
        q_pvalues = pd.Series(np.nan, index=Xq.columns)

    try:
        auc = roc_auc_score(y_bin, dat[p_col])
    except Exception:
        auc = np.nan

    threshold_errors = int(
        ((dat[p_col] > 0.5).astype(int).values != y_bin).sum()
    )

    if pd.notna(auc):
        q_complete_separation = (
            q_complete_separation
            or (auc == 1.0 and threshold_errors == 0)
            or ("Perfect separation" in q_note)
        )

    if q_complete_separation:
        q_usable_for_significance = False

    logit_q_hat = (
        q_params.get("const", 0.0)
        + q_params.get("logit_p_BNF", 0.0) * dat["logit_p_BNF"]
    )

    q_hat = 1 / (1 + np.exp(-np.clip(logit_q_hat, -30, 30)))
    q_hat = pd.Series(q_hat, index=dat.index, name="q_hat")

    eps = 1e-12
    q_loglik = np.sum(
        np.where(
            y_bin == 1,
            np.log(np.clip(q_hat.values, eps, 1 - eps)),
            np.log(np.clip(1 - q_hat.values, eps, 1 - eps))
        )
    )

    def q_row(parameter_name, variable_name, var_key):
        pval = float(q_pvalues.get(var_key, np.nan))

        return {
            "model": "Common q-model",
            "component": "inflation probability",
            "parameter": parameter_name,
            "equation_part": "q_it = logit^{-1}(alpha_p + beta_p logit(p_BNF))",
            "variable": variable_name,
            "coef": float(q_params.get(var_key, np.nan)),
            "se": float(q_bse.get(var_key, np.nan)),
            "z": float(q_z.get(var_key, np.nan)),
            "p_value": pval,
            "sig": "NA" if not q_usable_for_significance else sig_star(pval),
            "usable_for_significance": "No" if not q_usable_for_significance else "Yes",
            "note": (
                "complete separation or fallback model; Wald test not interpretable"
                if not q_usable_for_significance
                else q_note
            ),
        }

    q_rows = pd.DataFrame([
        q_row("alpha_p", "(Intercept)", "const"),
        q_row("beta_p", "logit_p_BNF", "logit_p_BNF"),
    ])

    q_info = {
        "q_model_AUC_p_mean": float(auc) if pd.notna(auc) else np.nan,
        "q_model_threshold_0.5_errors": int(threshold_errors),
        "q_model_complete_separation": str(q_complete_separation),
        "q_model_usable_for_significance": str(q_usable_for_significance),
        "q_model_loglik": float(q_loglik),
        "q_model_n_params": int(len(q_params)),
        "q_model_note": q_note,
    }

    return q_params, q_hat, q_loglik, q_rows, q_info


q_params, q_hat, q_loglik, q_rows, q_info = fit_q_model(dat, p_col=P_COL)
dat["q_hat"] = q_hat


# =========================================================
# 11) Fit normal component
# =========================================================
def fit_normal_component(
    dat,
    model_name,
    x_vars,
    v_vars,
    y_col,
    q_hat,
    q_loglik,
    point_mass_c,
):
    normal_dat = dat[dat["true_inflated"] == 0].copy()

    if len(normal_dat) <= 5:
        raise ValueError(f"{model_name}: 非膨脹樣本數過少，無法估計 normal component。")

    y = normal_dat[y_col].astype(float).values

    X_df = sm.add_constant(
        normal_dat[x_vars],
        has_constant="add"
    ).astype(float)

    if len(v_vars) > 0:
        Z_df = sm.add_constant(
            normal_dat[v_vars],
            has_constant="add"
        ).astype(float)
    else:
        Z_df = pd.DataFrame({"const": 1.0}, index=normal_dat.index)

    X = X_df.values
    Z = Z_df.values

    p = X.shape[1]
    k = Z.shape[1]

    beta_start = np.linalg.lstsq(X, y, rcond=None)[0]
    resid_start = y - X @ beta_start

    gamma_start = np.zeros(k)
    gamma_start[0] = np.log(np.var(resid_start) + 1e-6)

    theta_start = np.r_[beta_start, gamma_start]

    def nll(theta):
        beta = theta[:p]
        gamma = theta[p:]

        mu = X @ beta
        eta = np.clip(Z @ gamma, -20, 20)

        r = y - mu

        return 0.5 * np.sum(
            np.log(2 * np.pi)
            + eta
            + (r * r) * np.exp(-eta)
        )

    def grad(theta):
        beta = theta[:p]
        gamma = theta[p:]

        mu = X @ beta
        eta = np.clip(Z @ gamma, -20, 20)

        r = y - mu
        inv_var = np.exp(-eta)

        g_beta = -X.T @ (r * inv_var)
        g_gamma = 0.5 * Z.T @ (1 - (r * r) * inv_var)

        return np.r_[g_beta, g_gamma]

    opt = minimize(
        nll,
        theta_start,
        jac=grad,
        method="L-BFGS-B",
        options={
            "maxiter": 5000,
            "gtol": 1e-8,
            "ftol": 1e-12,
            "maxls": 50,
        },
    )

    H = approx_hess(opt.x, nll)
    cov = np.linalg.pinv(H)

    diag_cov = np.diag(cov)
    se = np.sqrt(np.where(diag_cov > 0, diag_cov, np.nan))

    beta_hat = opt.x[:p]
    gamma_hat = opt.x[p:]

    mu_hat_normal = X @ beta_hat
    eta_hat_normal = np.clip(Z @ gamma_hat, -20, 20)
    omega2_hat_normal = np.exp(eta_hat_normal)

    normal_resid = y - mu_hat_normal

    normal_loglik = float(-opt.fun)
    normal_mae = mae(y, mu_hat_normal)
    normal_rmse = rmse(y, mu_hat_normal)

    # Predict mu for all observations, not only non-inflated.
    X_all_df = sm.add_constant(
        dat[x_vars],
        has_constant="add"
    ).astype(float)

    if len(v_vars) > 0:
        Z_all_df = sm.add_constant(
            dat[v_vars],
            has_constant="add"
        ).astype(float)
    else:
        Z_all_df = pd.DataFrame({"const": 1.0}, index=dat.index)

    mu_hat_all = X_all_df.values @ beta_hat
    eta_hat_all = np.clip(Z_all_df.values @ gamma_hat, -20, 20)
    omega2_hat_all = np.exp(eta_hat_all)

    # Mixture mean prediction for all observations:
    # E[Y_it] = q_it * c + (1 - q_it) * mu_it
    mixture_mean_hat_all = (
        q_hat.values * point_mass_c
        + (1 - q_hat.values) * mu_hat_all
    )

    all_y = dat[y_col].astype(float).values
    mixture_mean_mae_all = mae(all_y, mixture_mean_hat_all)
    mixture_mean_rmse_all = rmse(all_y, mixture_mean_hat_all)

    total_loglik = float(q_loglik + normal_loglik)

    n_total = int(dat.shape[0])
    n_noninflated = int(normal_dat.shape[0])

    q_param_count = 2
    normal_param_count = int(p + k)
    total_param_count = int(q_param_count + normal_param_count)

    total_aic = float(2 * total_param_count - 2 * total_loglik)
    total_bic = float(np.log(n_total) * total_param_count - 2 * total_loglik)

    normal_aic = float(2 * normal_param_count - 2 * normal_loglik)
    normal_bic = float(np.log(n_noninflated) * normal_param_count - 2 * normal_loglik)

    # Coefficient summary
    names = []
    blocks = []
    components = []
    equation_parts = []

    for name in X_df.columns:
        if name == "const":
            blocks.append("beta_0")
            names.append("(Intercept)")
        elif name == F_COL:
            blocks.append("lambda")
            names.append(F_COL)
        else:
            blocks.append("beta_x")
            names.append(name)

        components.append("normal mean")
        equation_parts.append("mu_it = beta_0 + X beta_x" if F_COL not in x_vars else "mu_it = beta_0 + lambda F_it + X beta_x")

    for name in Z_df.columns:
        if name == "const":
            blocks.append("alpha_sigma")
            names.append("(Intercept)")
        else:
            blocks.append("beta_sigma")
            names.append(name)

        components.append("log variance")
        equation_parts.append("log(omega_it^2) = alpha_sigma + V beta_sigma")

    coef_df = pd.DataFrame({
        "model": model_name,
        "component": components,
        "parameter": blocks,
        "equation_part": equation_parts,
        "variable": names,
        "coef": opt.x,
        "se": se,
    })

    coef_df["z"] = coef_df["coef"] / coef_df["se"]
    coef_df["p_value"] = 2 * (1 - norm.cdf(np.abs(coef_df["z"])))
    coef_df["sig"] = coef_df["p_value"].apply(sig_star)
    coef_df["usable_for_significance"] = "Yes"
    coef_df["note"] = "non-inflated subset heteroscedastic normal MLE"

    # Fitted table
    # 重要：index 要保留 dat.index，否則後面用 normal_dat.index 回填殘差時會 KeyError。
    fitted_df = pd.DataFrame(
        {
            "model": model_name,
            "row_index": dat.index,
            "y": dat[y_col].astype(float).values,
            "true_inflated": dat["true_inflated"].astype(int).values,
            "q_hat": q_hat.loc[dat.index].values if isinstance(q_hat, pd.Series) else np.asarray(q_hat),
            "mu_hat": mu_hat_all,
            "log_omega2_hat": eta_hat_all,
            "omega2_hat": omega2_hat_all,
            "omega_hat": np.sqrt(omega2_hat_all),
            "mixture_mean_hat": mixture_mean_hat_all,
            "mixture_mean_resid": all_y - mixture_mean_hat_all,
        },
        index=dat.index
    )

    # Normal-only residual is meaningful only for non-inflated component.
    fitted_df["normal_component_resid"] = np.nan
    fitted_df.loc[normal_dat.index, "normal_component_resid"] = normal_resid

    model_metrics = {
        "model": model_name,
        "normal_mean_structure": (
            "mu_it = beta_0 + X_it beta_x"
            if F_COL not in x_vars
            else "mu_it = beta_0 + X_it beta_x + lambda F_it"
        ),
        "variance_structure": "log(omega_it^2) = alpha_sigma + V_it beta_sigma",
        "n_total": n_total,
        "n_inflated": int(dat["true_inflated"].sum()),
        "n_noninflated": n_noninflated,
        "n_mean_variables_without_intercept": int(len(x_vars)),
        "n_variance_variables_without_intercept": int(len(v_vars)),
        "normal_loglik": normal_loglik,
        "q_loglik": float(q_loglik),
        "total_mixture_loglik": total_loglik,
        "normal_component_MAE": normal_mae,
        "normal_component_RMSE": normal_rmse,
        "mixture_mean_MAE_all": mixture_mean_mae_all,
        "mixture_mean_RMSE_all": mixture_mean_rmse_all,
        "normal_component_AIC": normal_aic,
        "normal_component_BIC": normal_bic,
        "total_mixture_AIC": total_aic,
        "total_mixture_BIC": total_bic,
        "q_params": q_param_count,
        "normal_params": normal_param_count,
        "total_params": total_param_count,
        "optimizer_success": str(opt.success),
        "optimizer_message": str(opt.message),
    }

    return coef_df, fitted_df, model_metrics


coef_x, fitted_x, metrics_x = fit_normal_component(
    dat=dat,
    model_name="MIX-X: specified covariates only",
    x_vars=X_ONLY_VARS,
    v_vars=V_vars,
    y_col=Y_COL,
    q_hat=q_hat,
    q_loglik=q_loglik,
    point_mass_c=POINT_MASS_C,
)

coef_xf, fitted_xf, metrics_xf = fit_normal_component(
    dat=dat,
    model_name="MIX-XF: specified covariates + F_it",
    x_vars=X_F_VARS,
    v_vars=V_vars,
    y_col=Y_COL,
    q_hat=q_hat,
    q_loglik=q_loglik,
    point_mass_c=POINT_MASS_C,
)


# =========================================================
# 12) Model comparison
# =========================================================
model_comparison = pd.DataFrame([metrics_x, metrics_xf])

# Add improvement row: MIX-X minus MIX-XF
improvement = {
    "model": "Improvement: MIX-X minus MIX-XF",
    "normal_mean_structure": "positive values mean MIX-XF is better for error/AIC/BIC",
    "variance_structure": "same as above",
    "n_total": np.nan,
    "n_inflated": np.nan,
    "n_noninflated": np.nan,
    "n_mean_variables_without_intercept": np.nan,
    "n_variance_variables_without_intercept": np.nan,
    "normal_loglik": metrics_xf["normal_loglik"] - metrics_x["normal_loglik"],
    "q_loglik": 0.0,
    "total_mixture_loglik": metrics_xf["total_mixture_loglik"] - metrics_x["total_mixture_loglik"],
    "normal_component_MAE": metrics_x["normal_component_MAE"] - metrics_xf["normal_component_MAE"],
    "normal_component_RMSE": metrics_x["normal_component_RMSE"] - metrics_xf["normal_component_RMSE"],
    "mixture_mean_MAE_all": metrics_x["mixture_mean_MAE_all"] - metrics_xf["mixture_mean_MAE_all"],
    "mixture_mean_RMSE_all": metrics_x["mixture_mean_RMSE_all"] - metrics_xf["mixture_mean_RMSE_all"],
    "normal_component_AIC": metrics_x["normal_component_AIC"] - metrics_xf["normal_component_AIC"],
    "normal_component_BIC": metrics_x["normal_component_BIC"] - metrics_xf["normal_component_BIC"],
    "total_mixture_AIC": metrics_x["total_mixture_AIC"] - metrics_xf["total_mixture_AIC"],
    "total_mixture_BIC": metrics_x["total_mixture_BIC"] - metrics_xf["total_mixture_BIC"],
    "q_params": np.nan,
    "normal_params": np.nan,
    "total_params": np.nan,
    "optimizer_success": "",
    "optimizer_message": "",
}

model_comparison = pd.concat(
    [model_comparison, pd.DataFrame([improvement])],
    ignore_index=True
)


# =========================================================
# 13) Coefficient tables
# =========================================================
coef_table = pd.concat(
    [
        q_rows,
        coef_x,
        coef_xf,
    ],
    ignore_index=True
)

significant_only = coef_table[
    (coef_table["sig"].isin(["***", "**", "*", "."])) |
    (coef_table["parameter"].isin(["alpha_p", "beta_p", "lambda"]))
].copy()


# =========================================================
# 14) Model information
# =========================================================
model_info_records = [
    ["source_data_file", str(DATA_FILE)],
    ["source_sheet", RESULT_SHEET],
    ["outcome_column", Y_COL],
    ["point_mass_c", POINT_MASS_C],
    ["n_rows_after_dropna", int(dat.shape[0])],
    ["n_inflated", int(dat["true_inflated"].sum())],
    ["n_noninflated", int((dat["true_inflated"] == 0).sum())],
    ["n_continuous_z_variables", int(len(continuous_z_cols))],
    ["n_proportion_logit_z_variables", int(len(proportion_logit_z_cols))],
    ["n_MIX_X_mean_variables", int(len(X_ONLY_VARS))],
    ["n_MIX_XF_mean_variables", int(len(X_F_VARS))],
    ["n_variance_variables", int(len(V_vars))],
]

for k, v in q_info.items():
    model_info_records.append([k, v])

model_info = pd.DataFrame(model_info_records, columns=["item", "value"])


# =========================================================
# 15) Variable design tables
# =========================================================
variable_design_records = []

for model_name, x_vars in [
    ("MIX-X: specified covariates only", X_ONLY_VARS),
    ("MIX-XF: specified covariates + F_it", X_F_VARS),
]:
    for c in x_vars:
        if c == F_COL:
            raw_name = F_COL
            role = "normal_mean_BayesNF_F"
            transformation = "none"
            standardized = False
        elif c in continuous_z_cols:
            raw_name = c.replace("_z", "")
            role = "normal_mean_continuous_X"
            transformation = "x -> z-score"
            standardized = True
        elif c in proportion_logit_z_cols:
            raw_name = c.replace("_logit_z", "")
            role = "normal_mean_proportion_X"
            transformation = "p -> logit(p) -> z-score"
            standardized = True
        else:
            raw_name = c
            role = "normal_mean_X"
            transformation = "unknown"
            standardized = False

        variable_design_records.append({
            "model": model_name,
            "equation": "normal_mean",
            "variable_used_in_model": c,
            "raw_variable": raw_name,
            "role": role,
            "transformation": transformation,
            "standardized": standardized,
        })

for c in V_vars:
    if c in continuous_z_cols:
        raw_name = c.replace("_z", "")
        transformation = "x -> z-score"
        standardized = True
    elif c in proportion_logit_z_cols:
        raw_name = c.replace("_logit_z", "")
        transformation = "p -> logit(p) -> z-score"
        standardized = True
    elif c in ["high_risk", "potential_high_risk"]:
        raw_name = c
        transformation = "raw 0/1 indicator"
        standardized = False
    else:
        raw_name = c
        transformation = "unknown"
        standardized = False

    variable_design_records.append({
        "model": "both MIX-X and MIX-XF",
        "equation": "log_variance",
        "variable_used_in_model": c,
        "raw_variable": raw_name,
        "role": "log_variance_V",
        "transformation": transformation,
        "standardized": standardized,
    })

variable_design_table = pd.DataFrame(variable_design_records)

reference_table = pd.DataFrame({
    "removed_reference_variable": reference_cols,
    "reason": "reference group to avoid perfect multicollinearity"
})

continuous_transform_table = transform_map[
    transform_map["variable_type"] == "specified_continuous"
].copy()

proportion_transform_table = transform_map[
    transform_map["variable_type"] == "non_specified_continuous_treated_as_proportion"
].copy()

fitted_values = pd.concat(
    [fitted_x, fitted_xf],
    ignore_index=True
)


# =========================================================
# 16) Export
# =========================================================
with pd.ExcelWriter(OUT_FILE, engine="openpyxl") as writer:
    model_comparison.to_excel(writer, sheet_name="Model_Comparison", index=False)
    coef_table.to_excel(writer, sheet_name="Coefficient_Table", index=False)
    significant_only.to_excel(writer, sheet_name="Significant_Only", index=False)
    model_info.to_excel(writer, sheet_name="Model_Info", index=False)
    transform_map.to_excel(writer, sheet_name="Transform_Map", index=False)
    continuous_transform_table.to_excel(writer, sheet_name="Continuous_Z_Params", index=False)
    proportion_transform_table.to_excel(writer, sheet_name="Proportion_Logit_Z", index=False)
    variable_design_table.to_excel(writer, sheet_name="Variable_Design", index=False)
    reference_table.to_excel(writer, sheet_name="Reference_Cols", index=False)
    dat.to_excel(writer, sheet_name="Model_Used_Data", index=False)
    fitted_values.to_excel(writer, sheet_name="Fitted_Values", index=False)


# =========================================================
# 17) Print results
# =========================================================
print("\n==============================")
print("Model comparison")
print("==============================")
print(model_comparison)

print("\n==============================")
print("Coefficient table")
print("==============================")
print(coef_table)

print("\n==============================")
print("Significant only")
print("==============================")
print(significant_only)

print("\n==============================")
print("Mean equation variables: MIX-X")
print("==============================")
print(X_ONLY_VARS)

print("\n==============================")
print("Mean equation variables: MIX-XF")
print("==============================")
print(X_F_VARS)

print("\n==============================")
print("Variance equation variables")
print("==============================")
print(V_vars)

print("\n==============================")
print("Continuous z variables")
print("==============================")
print(continuous_z_cols)

print("\n==============================")
print("Proportion logit-z variables")
print("==============================")
print(proportion_logit_z_cols)

print(f"\nSaved: {OUT_FILE}")

資料筆數： 1297
資料欄位數： 90
轉換前可用資料筆數： 1290
膨脹點筆數： 422
非膨脹點筆數： 868
指定連續型 z-score 變數數量： 13
比例變數 logit-z 變數數量： 20
進入 mixture model 的資料筆數： 1290
進入 mixture model 的膨脹點筆數： 422
進入 mixture model 的非膨脹點筆數： 868
MIX-X mean variables 數量： 33
MIX-XF mean variables 數量： 34
Variance equation V_vars 數量： 9

Model comparison
                                 model  \
0     MIX-X: specified covariates only   
1  MIX-XF: specified covariates + F_it   
2      Improvement: MIX-X minus MIX-XF   

                               normal_mean_structure  \
0                       mu_it = beta_0 + X_it beta_x   
1         mu_it = beta_0 + X_it beta_x + lambda F_it   
2  positive values mean MIX-XF is better for erro...   

                                variance_structure  n_total  n_inflated  \
0  log(omega_it^2) = alpha_sigma + V_it beta_sigma   1290.0       422.0   
1  log(omega_it^2) = alpha_sigma + V_it beta_sigma   1290.0       422.0   
2                                    same as above      NaN         NaN   

   n_n

In [5]:
# -*- coding: utf-8 -*-
"""
Fit the inflated-normal mixture regression framework:

Y_it ~ q_it delta_c + (1-q_it) N(mu_it, omega_it^2)

Inflation probability model:
q_it = logit^{-1}[alpha_p + beta_p logit(p_BNF_it)]

Normal mean model for non-inflated observations:
mu_it = beta_0 + lambda F_BNF_it + X_it^T beta_x

Log variance model for non-inflated observations:
log(omega_it^2) = alpha_sigma + V_it^T beta_sigma


本版資料與變數設計：

1. 直接讀取 BayesNF 結果檔 row_level。
   不再另外讀 raw file，也不再做 merge。

2. 指定連續型變數：
   x -> z-score

3. 其他非指定連續型變數：
   視為比例變數
   p -> logit(p) -> z-score

4. F_mean 保留原尺度，作為 BayesNF 平均場輸入。

5. p_mean 轉為 logit_p_BNF，作為膨脹機率校準模型輸入。

6. variance equation 預設使用部分已轉換後變數，
   並保留 high_risk / potential_high_risk 作為 0/1 indicator。
"""

import warnings
import numpy as np
import pandas as pd
import statsmodels.api as sm

from pathlib import Path
from scipy.optimize import minimize
from scipy.stats import norm
from statsmodels.tools.numdiff import approx_hess
from statsmodels.tools.sm_exceptions import PerfectSeparationError
from sklearn.metrics import roc_auc_score


# =========================================================
# 0) File paths
# =========================================================
DATA_FILE = Path(
    r"C:/Users/User/Desktop/LC analysis/result/inflated_normal_highrisk_five_features_results.xlsx"
)

OUT_FILE = Path(
    r"C:/Users/User/Desktop/LC analysis/result/inflated_normal_mixture_coefficients_logit_prop.xlsx"
)

RESULT_SHEET = "row_level"


# =========================================================
# 1) Basic utilities
# =========================================================
def unique(seq):
    out, seen = [], set()
    for x in seq:
        if x not in seen:
            out.append(x)
            seen.add(x)
    return out


def sig_star(p):
    if pd.isna(p):
        return ""
    if p < 0.001:
        return "***"
    if p < 0.01:
        return "**"
    if p < 0.05:
        return "*"
    if p < 0.10:
        return "."
    return ""


def parse_year(s):
    s = pd.Series(s).copy()
    numeric = pd.to_numeric(s, errors="coerce")

    out = pd.Series(index=s.index, dtype="float")

    mask_excel_date = numeric.between(30000, 60000)
    out.loc[mask_excel_date] = pd.to_datetime(
        numeric.loc[mask_excel_date],
        unit="D",
        origin="1899-12-30",
        errors="coerce"
    ).dt.year

    mask_year = numeric.between(1900, 2100)
    out.loc[mask_year] = numeric.loc[mask_year]

    mask_rest = out.isna()
    out.loc[mask_rest] = pd.to_datetime(
        s.loc[mask_rest],
        errors="coerce"
    ).dt.year

    return out.astype("Int64")


def clean_town(x):
    return (
        x.astype(str)
        .str.strip()
        .str.replace("臺", "台", regex=False)
        .str.replace(" ", "", regex=False)
    )


def zscore_series(s):
    s = pd.to_numeric(s, errors="coerce")
    mean_s = s.mean()
    sd_s = s.std(ddof=0)

    if pd.isna(sd_s) or sd_s <= 0:
        return None, mean_s, sd_s

    return (s - mean_s) / sd_s, mean_s, sd_s


def safe_logit01(s, eps=1e-6):
    """
    Input must be probability scale.
    Clip p into [eps, 1-eps], then apply logit.
    """
    s = pd.to_numeric(s, errors="coerce")
    s_clip = np.clip(s, eps, 1 - eps)
    return np.log(s_clip / (1 - s_clip))


def proportion_to_01(s, var_name):
    """
    將比例變數轉成 [0,1] 尺度。

    可接受：
    1. 原本就是比例，例如 0.32
    2. 百分比格式，例如 32.0，會自動除以 100

    若數值不在 [0,1] 或 [0,100]，會報錯提醒。
    """
    s = pd.to_numeric(s, errors="coerce")

    min_s = s.min(skipna=True)
    max_s = s.max(skipna=True)

    if pd.isna(min_s) or pd.isna(max_s):
        raise ValueError(f"{var_name} 無法判斷比例範圍，可能全為缺失值。")

    if min_s >= 0 and max_s <= 1:
        p = s.copy()
        scale_type = "0_to_1"

    elif min_s >= 0 and max_s <= 100:
        p = s / 100
        scale_type = "0_to_100_divided_by_100"

    else:
        raise ValueError(
            f"{var_name} 被設定為比例變數，但數值範圍為 "
            f"[{min_s}, {max_s}]，不在 [0,1] 或 [0,100] 內。"
        )

    return p, scale_type


def safe_logit_with_clip_count(p, eps=1e-6):
    p = pd.to_numeric(p, errors="coerce")

    n_lower_clip = int((p <= eps).sum())
    n_upper_clip = int((p >= 1 - eps).sum())

    p_clip = p.clip(lower=eps, upper=1 - eps)
    logit_value = np.log(p_clip / (1 - p_clip))

    return logit_value, p_clip, n_lower_clip, n_upper_clip


# =========================================================
# 2) Read data
# =========================================================
df = pd.read_excel(DATA_FILE, sheet_name=RESULT_SHEET)

print("資料筆數：", len(df))
print("資料欄位數：", df.shape[1])

# 若 Excel 讀進來有重複欄位名稱，可視情況補正。
# 如果你的檔案已經是 癌症期別1、癌症期別2 等欄位，這段不會影響。
df = df.rename(columns={
    "癌症期別": "癌症期別1",
    "...17": "癌症期別2",
    "癌症組織類型": "癌症組織類型1",
    "...21": "癌症組織類型2",
    "...22": "癌症組織類型3",
    "...23": "癌症組織類型4",
})

df = df.copy()

if "年份" in df.columns:
    df["year"] = parse_year(df["年份"])

if "鄉鎮" in df.columns:
    df["鄉鎮_key"] = clean_town(df["鄉鎮"])
    df["鄉鎮6"] = df["鄉鎮_key"].str[:6]


# =========================================================
# 3) Define outcome, point mass, and BayesNF outputs
# =========================================================
POINT_MASS_C = -6.907255070523717

# 這裡優先使用 死亡_cll。
# 若你的 row_level 裡面沒有死亡_cll，則依序改用 y_imputed 或 y_true。
if "死亡_cll" in df.columns:
    Y_COL = "死亡_cll"
elif "y_imputed" in df.columns:
    Y_COL = "y_imputed"
elif "y_true" in df.columns:
    Y_COL = "y_true"
else:
    raise ValueError("資料缺少反應變數欄位：死亡_cll、y_imputed、y_true 都不存在。")

F_COL = "F_mean"
P_COL = "p_mean"

for c in [Y_COL, F_COL, P_COL]:
    if c not in df.columns:
        raise ValueError(f"資料缺少必要欄位：{c}")

df[Y_COL] = pd.to_numeric(df[Y_COL], errors="coerce")
df[F_COL] = pd.to_numeric(df[F_COL], errors="coerce")
df[P_COL] = pd.to_numeric(df[P_COL], errors="coerce")

# true_inflated 若不存在，則由反應值是否等於膨脹點判定。
if "true_inflated" not in df.columns:
    df["true_inflated"] = (
        np.abs(pd.to_numeric(df[Y_COL], errors="coerce") - POINT_MASS_C) < 1e-8
    ).astype(int)
else:
    df["true_inflated"] = pd.to_numeric(df["true_inflated"], errors="coerce").astype("Int64")


# =========================================================
# 4) Covariates
#    變數清單與上一版三種回歸保持一致。
# =========================================================
covariates = [
    "該鄉鎮總人口數 (人數)",
    # "小於18歲人口數 (人數)",
    "Lead_water",
    "Lead_soil",
    "PM2.5 (ug/m3)",
    "Income_regular (百萬元)",
    "每百萬人病床數 (床/萬人)",
    "平均年齡（歲）",
    "性別(男)",
    "平均BMI  (kg/m2)",
    "平均日常體能狀態（分）",
    "加權 CCI",
    "癌症期別1",
    "癌症期別2",
    "仍吸菸",
    "仍喝酒",
    "癌症組織類型1",
    "癌症組織類型2",
    "癌症組織類型3",
    "癌症組織類型4",
    "OP指標日期前",
    "RT指標日期後",
    "併用藥物1",
    "併用藥物2",
    "併用藥物3",
    "併用藥物4",
    "併用藥物5",
    "併用藥物6",
    "併用藥物7",
    "併用藥物8",
    "併用藥物9",
    "併用藥物10",
    "併用藥物11",
    "平均診斷到index date時間(日)",
    "平均健保投保金額",
    "醫學中心",
]

covariates = [c for c in covariates if c in df.columns]

# 移除 reference category，避免完全共線性。
reference_cols = [
    "癌症期別2",
    "癌症組織類型4",
]
reference_cols = [c for c in reference_cols if c in covariates]

x_raw_cols = [c for c in covariates if c not in reference_cols]

# 避免 outcome 或 BayesNF 輸出被放進一般 X
leakage_cols = [
    "死亡",
    "死亡_cll",
    "死亡_logit",
    "死亡_probit",
    "death_count",
    "death_count_raw",
    "y_true",
    "y_imputed",
    "F_mean",
    "p_mean",
    "sigma_mean",
    "sigma2_mean",
    "F_q05",
    "F_q95",
    "p_q05",
    "p_q95",
    "sigma_q05",
    "sigma_q95",
    "sigma2_q05",
    "sigma2_q95",
    "yhat_mix_mean",
    "yhat_mix_q05",
    "yhat_mix_q95",
    "err_mix",
    "abs_err_mix",
    "sq_err_mix",
    "err_F_only",
    "abs_err_F_only",
    "sq_err_F_only",
    "pred_inflated_05",
    "true_inflated",
]

x_raw_cols = [c for c in x_raw_cols if c not in leakage_cols]

for c in x_raw_cols:
    df[c] = pd.to_numeric(df[c], errors="coerce")

# 移除沒有變異的 X
x_raw_cols = [
    c for c in x_raw_cols
    if c in df.columns and df[c].nunique(dropna=True) > 1
]


# =========================================================
# 5) 指定連續型與比例型變數
#
# 指定連續型：
# x -> z-score
#
# 非指定連續型：
# p -> logit(p) -> z-score
# =========================================================
CONTINUOUS_COLS = [
    "該鄉鎮總人口數 (人數)",
    "Lead_water",
    "Lead_soil",
    "PM2.5 (ug/m3)",
    "Income_regular (百萬元)",
    "每百萬人病床數 (床/萬人)",
    "平均年齡（歲）",
    "性別(男)",
    "平均BMI  (kg/m2)",
    "平均日常體能狀態（分）",
    "加權 CCI",
    "平均診斷到index date時間(日)",
    "平均健保投保金額",
]

continuous_cols = [
    c for c in CONTINUOUS_COLS
    if c in x_raw_cols and c in df.columns
]

proportion_cols = [
    c for c in x_raw_cols
    if c not in continuous_cols
]


# =========================================================
# 6) Build regression sample
# =========================================================
id_cols = [
    c for c in ["年份", "year", "鄉鎮", "鄉鎮_key", "鄉鎮6"]
    if c in df.columns
]

needed_raw = unique(
    id_cols
    + [Y_COL, "true_inflated", F_COL, P_COL]
    + x_raw_cols
    + [c for c in ["high_risk", "potential_high_risk"] if c in df.columns]
)

tf = (
    df[needed_raw]
    .replace([np.inf, -np.inf], np.nan)
    .copy()
)

# 先只針對必要原始欄位 dropna。
# 後面轉換後會再 dropna 一次。
tf = tf.dropna(subset=[Y_COL, "true_inflated", F_COL, P_COL] + x_raw_cols).copy()
tf["true_inflated"] = tf["true_inflated"].astype(int)

print("轉換前可用資料筆數：", len(tf))
print("膨脹點筆數：", int(tf["true_inflated"].sum()))
print("非膨脹點筆數：", int((tf["true_inflated"] == 0).sum()))


# =========================================================
# 7) Transform BNF p_mean
# =========================================================
tf["logit_p_BNF"] = safe_logit01(tf[P_COL], eps=1e-6)


# =========================================================
# 8) Transform covariates
# =========================================================
transform_records = []
transformed_cols = {}

continuous_z_cols = []
proportion_logit_z_cols = []

# A) 指定連續型：x -> z-score
for c in continuous_cols:
    z_col = f"{c}_z"

    z_value, mean_c, sd_c = zscore_series(tf[c])

    if z_value is not None:
        tf[z_col] = z_value
        transformed_cols[c] = z_col
        continuous_z_cols.append(z_col)

        transform_records.append({
            "raw_variable": c,
            "model_variable": z_col,
            "variable_type": "specified_continuous",
            "transformation": "x -> z-score",
            "proportion_scale_type": "",
            "mean_used_for_z": mean_c,
            "sd_used_for_z": sd_c,
            "epsilon_for_logit": np.nan,
            "n_lower_clipped": np.nan,
            "n_upper_clipped": np.nan,
        })
    else:
        print(f"連續變數標準化略過：{c}，因為標準差為 0 或無法計算。")


# B) 非指定連續型：p -> logit(p) -> z-score
for c in proportion_cols:
    prop_col = f"{c}_prop01"
    logit_col = f"{c}_logit"
    logit_z_col = f"{c}_logit_z"

    p_raw, scale_type = proportion_to_01(tf[c], c)
    logit_value, p_clip, n_lower_clip, n_upper_clip = safe_logit_with_clip_count(
        p_raw,
        eps=1e-6
    )

    tf[prop_col] = p_clip
    tf[logit_col] = logit_value

    z_value, mean_logit, sd_logit = zscore_series(tf[logit_col])

    if z_value is not None:
        tf[logit_z_col] = z_value
        transformed_cols[c] = logit_z_col
        proportion_logit_z_cols.append(logit_z_col)

        transform_records.append({
            "raw_variable": c,
            "model_variable": logit_z_col,
            "variable_type": "non_specified_continuous_treated_as_proportion",
            "transformation": "p -> logit(p) -> z-score",
            "proportion_scale_type": scale_type,
            "mean_used_for_z": mean_logit,
            "sd_used_for_z": sd_logit,
            "epsilon_for_logit": 1e-6,
            "n_lower_clipped": n_lower_clip,
            "n_upper_clipped": n_upper_clip,
        })
    else:
        print(f"比例變數 logit 後標準化略過：{c}，因為 logit 後標準差為 0 或無法計算。")


transform_map = pd.DataFrame(transform_records)

print("指定連續型 z-score 變數數量：", len(continuous_z_cols))
print("比例變數 logit-z 變數數量：", len(proportion_logit_z_cols))


# =========================================================
# 9) Choose X and V
# =========================================================
# Mean equation:
# mu_it = beta_0 + lambda F_BNF_it + X_it beta_x
X_vars = unique(
    [F_COL]
    + continuous_z_cols
    + proportion_logit_z_cols
)

# Variance equation:
# log(omega_it^2) = alpha_sigma + V_it beta_sigma
#
# 預設保留原本 mixture code 的「較小 V 結構」概念，
# 但變數名稱改成目前轉換後的版本。
V_raw_candidates = [
    "Lead_water",
    "Lead_soil",
    "PM2.5 (ug/m3)",
    "每百萬人病床數 (床/萬人)",
    "平均年齡（歲）",
    "平均日常體能狀態（分）",
    "加權 CCI",
    "平均診斷到index date時間(日)",
]

V_vars = []
for raw_c in V_raw_candidates:
    if raw_c in transformed_cols:
        V_vars.append(transformed_cols[raw_c])

# high_risk / potential_high_risk 是風險標記，不視為比例變數轉 logit。
# 若存在，保留原始 0/1 indicator 放入 variance equation。
for risk_col in ["high_risk", "potential_high_risk"]:
    if risk_col in tf.columns:
        tf[risk_col] = pd.to_numeric(tf[risk_col], errors="coerce")
        if tf[risk_col].nunique(dropna=True) > 1:
            V_vars.append(risk_col)

V_vars = unique([v for v in V_vars if v in tf.columns])

needed_model = unique(
    [Y_COL, "true_inflated", P_COL, "logit_p_BNF"]
    + X_vars
    + V_vars
)

dat = (
    tf[needed_model]
    .replace([np.inf, -np.inf], np.nan)
    .dropna()
    .copy()
)

dat["true_inflated"] = dat["true_inflated"].astype(int)

print("進入 mixture model 的資料筆數：", len(dat))
print("進入 mixture model 的膨脹點筆數：", int(dat["true_inflated"].sum()))
print("進入 mixture model 的非膨脹點筆數：", int((dat["true_inflated"] == 0).sum()))
print("Mean equation X_vars 數量：", len(X_vars))
print("Variance equation V_vars 數量：", len(V_vars))


# =========================================================
# 10) q-model: alpha_p, beta_p
#
# q_it = logit^{-1}[alpha_p + beta_p logit(p_BNF_it)]
# =========================================================
y_bin = dat["true_inflated"].astype(int).values
Xq = sm.add_constant(dat[["logit_p_BNF"]], has_constant="add")

q_note = ""
q_complete_separation = False
q_usable_for_significance = True

try:
    with warnings.catch_warnings(record=True) as w:
        warnings.simplefilter("always")
        q_res = sm.Logit(y_bin, Xq).fit(disp=False, maxiter=200)
        q_note = "; ".join(sorted(set([str(ww.message) for ww in w])))

    q_params = q_res.params
    q_bse = q_res.bse
    q_z = q_res.tvalues
    q_pvalues = q_res.pvalues

except PerfectSeparationError as e:
    q_complete_separation = True
    q_usable_for_significance = False
    q_note = f"PerfectSeparationError: {str(e)}"

    # fallback: regularized GLM Binomial，僅用於得到穩定係數。
    glm_binom = sm.GLM(y_bin, Xq, family=sm.families.Binomial())
    q_res_reg = glm_binom.fit_regularized(alpha=1e-6, L1_wt=0.0, maxiter=1000)

    q_params = pd.Series(q_res_reg.params, index=Xq.columns)
    q_bse = pd.Series(np.nan, index=Xq.columns)
    q_z = pd.Series(np.nan, index=Xq.columns)
    q_pvalues = pd.Series(np.nan, index=Xq.columns)

except Exception as e:
    q_complete_separation = True
    q_usable_for_significance = False
    q_note = f"Logit failed and fallback used: {str(e)}"

    glm_binom = sm.GLM(y_bin, Xq, family=sm.families.Binomial())
    q_res_reg = glm_binom.fit_regularized(alpha=1e-6, L1_wt=0.0, maxiter=1000)

    q_params = pd.Series(q_res_reg.params, index=Xq.columns)
    q_bse = pd.Series(np.nan, index=Xq.columns)
    q_z = pd.Series(np.nan, index=Xq.columns)
    q_pvalues = pd.Series(np.nan, index=Xq.columns)


# AUC
try:
    auc = roc_auc_score(y_bin, dat[P_COL])
except Exception:
    auc = np.nan

threshold_errors = int(
    ((dat[P_COL] > 0.5).astype(int).values != y_bin).sum()
)

if pd.notna(auc):
    q_complete_separation = (
        q_complete_separation
        or (auc == 1.0 and threshold_errors == 0)
        or ("Perfect separation" in q_note)
    )

if q_complete_separation:
    q_usable_for_significance = False


def q_row(parameter_name, variable_name, var_key):
    pval = float(q_pvalues.get(var_key, np.nan))

    return {
        "component": "inflation probability",
        "parameter": parameter_name,
        "equation_part": "q_it = logit^{-1}(alpha_p + beta_p logit(p_BNF))",
        "variable": variable_name,
        "coef": float(q_params.get(var_key, np.nan)),
        "se": float(q_bse.get(var_key, np.nan)),
        "z": float(q_z.get(var_key, np.nan)),
        "p_value": pval,
        "sig": "NA" if not q_usable_for_significance else sig_star(pval),
        "usable_for_significance": "No" if not q_usable_for_significance else "Yes",
        "note": (
            "complete separation or fallback model; Wald test not interpretable"
            if not q_usable_for_significance
            else q_note
        ),
    }


q_rows = pd.DataFrame([
    q_row("alpha_p", "(Intercept)", "const"),
    q_row("beta_p", "logit_p_BNF", "logit_p_BNF"),
])


# =========================================================
# 11) Non-inflated heteroscedastic normal MLE
#
# Only y != c, equivalently true_inflated == 0
#
# mu_it = beta_0 + lambda F_BNF + X beta_x
# log(omega_it^2) = alpha_sigma + V beta_sigma
# =========================================================
normal_dat = dat[dat["true_inflated"] == 0].copy()

if len(normal_dat) <= 5:
    raise ValueError("非膨脹樣本數過少，無法估計 normal component。")

y = normal_dat[Y_COL].astype(float).values

X_df = sm.add_constant(
    normal_dat[X_vars],
    has_constant="add"
).astype(float)

Z_df = sm.add_constant(
    normal_dat[V_vars],
    has_constant="add"
).astype(float)

X = X_df.values
Z = Z_df.values

p = X.shape[1]
k = Z.shape[1]

# OLS start for beta
beta_start = np.linalg.lstsq(X, y, rcond=None)[0]
resid_start = y - X @ beta_start

# log variance start for gamma
gamma_start = np.zeros(k)
gamma_start[0] = np.log(np.var(resid_start) + 1e-6)

theta_start = np.r_[beta_start, gamma_start]


def nll(theta):
    beta = theta[:p]
    gamma = theta[p:]

    mu = X @ beta
    eta = np.clip(Z @ gamma, -20, 20)  # eta = log(sigma^2)

    r = y - mu

    return 0.5 * np.sum(
        np.log(2 * np.pi)
        + eta
        + (r * r) * np.exp(-eta)
    )


def grad(theta):
    beta = theta[:p]
    gamma = theta[p:]

    mu = X @ beta
    eta = np.clip(Z @ gamma, -20, 20)

    r = y - mu
    inv_var = np.exp(-eta)

    g_beta = -X.T @ (r * inv_var)
    g_gamma = 0.5 * Z.T @ (1 - (r * r) * inv_var)

    return np.r_[g_beta, g_gamma]


opt = minimize(
    nll,
    theta_start,
    jac=grad,
    method="L-BFGS-B",
    options={
        "maxiter": 5000,
        "gtol": 1e-8,
        "ftol": 1e-12,
        "maxls": 50,
    },
)

# Numerical Hessian for standard errors
H = approx_hess(opt.x, nll)
cov = np.linalg.pinv(H)

diag_cov = np.diag(cov)
se = np.sqrt(np.where(diag_cov > 0, diag_cov, np.nan))


# =========================================================
# 12) Summarize normal component coefficients
# =========================================================
names = []
blocks = []

for name in X_df.columns:
    if name == "const":
        blocks.append("beta_0")
        names.append("(Intercept)")
    elif name == F_COL:
        blocks.append("lambda")
        names.append(F_COL)
    else:
        blocks.append("beta_x")
        names.append(name)

for name in Z_df.columns:
    if name == "const":
        blocks.append("alpha_sigma")
        names.append("(Intercept)")
    else:
        blocks.append("beta_sigma")
        names.append(name)

normal_summary = pd.DataFrame({
    "block": blocks,
    "variable": names,
    "coef": opt.x,
    "se": se,
})

normal_summary["z"] = normal_summary["coef"] / normal_summary["se"]
normal_summary["p_value"] = 2 * (1 - norm.cdf(np.abs(normal_summary["z"])))
normal_summary["sig"] = normal_summary["p_value"].apply(sig_star)

normal_rows = []

for _, r in normal_summary.iterrows():
    if r["block"] in ["beta_0", "lambda", "beta_x"]:
        component = "normal mean"
        equation_part = "mu_it = beta_0 + lambda F_BNF + X beta_x"
    else:
        component = "log variance"
        equation_part = "log(omega_it^2) = alpha_sigma + V beta_sigma"

    normal_rows.append({
        "component": component,
        "parameter": r["block"],
        "equation_part": equation_part,
        "variable": r["variable"],
        "coef": float(r["coef"]),
        "se": float(r["se"]),
        "z": float(r["z"]),
        "p_value": float(r["p_value"]),
        "sig": r["sig"],
        "usable_for_significance": "Yes",
        "note": "non-inflated subset heteroscedastic normal MLE",
    })


coef_table = pd.concat(
    [q_rows, pd.DataFrame(normal_rows)],
    ignore_index=True
)


# =========================================================
# 13) Fitted values and diagnostics
# =========================================================
# q fitted
logit_q_hat = q_params.get("const", 0.0) + q_params.get("logit_p_BNF", 0.0) * dat["logit_p_BNF"]
dat["q_hat"] = 1 / (1 + np.exp(-np.clip(logit_q_hat, -30, 30)))

# normal component fitted only for non-inflated data
normal_dat = normal_dat.copy()

beta_hat = opt.x[:p]
gamma_hat = opt.x[p:]

normal_dat["mu_hat"] = X @ beta_hat
normal_dat["log_omega2_hat"] = np.clip(Z @ gamma_hat, -20, 20)
normal_dat["omega2_hat"] = np.exp(normal_dat["log_omega2_hat"])
normal_dat["omega_hat"] = np.sqrt(normal_dat["omega2_hat"])
normal_dat["normal_resid"] = y - normal_dat["mu_hat"]

# put fitted normal values back
dat["mu_hat"] = np.nan
dat["log_omega2_hat"] = np.nan
dat["omega2_hat"] = np.nan
dat["omega_hat"] = np.nan
dat["normal_resid"] = np.nan

dat.loc[normal_dat.index, "mu_hat"] = normal_dat["mu_hat"]
dat.loc[normal_dat.index, "log_omega2_hat"] = normal_dat["log_omega2_hat"]
dat.loc[normal_dat.index, "omega2_hat"] = normal_dat["omega2_hat"]
dat.loc[normal_dat.index, "omega_hat"] = normal_dat["omega_hat"]
dat.loc[normal_dat.index, "normal_resid"] = normal_dat["normal_resid"]


# =========================================================
# 14) Model information
# =========================================================
model_info = pd.DataFrame([
    ["source_data_file", str(DATA_FILE)],
    ["source_sheet", RESULT_SHEET],
    ["outcome_column", Y_COL],
    ["point_mass_c", POINT_MASS_C],
    ["n_rows_after_dropna", int(dat.shape[0])],
    ["n_inflated", int(dat["true_inflated"].sum())],
    ["n_noninflated", int((dat["true_inflated"] == 0).sum())],
    ["q_model_AUC_p_mean", float(auc) if pd.notna(auc) else np.nan],
    ["q_model_threshold_0.5_errors", int(threshold_errors)],
    ["q_model_complete_separation", str(q_complete_separation)],
    ["q_model_usable_for_significance", str(q_usable_for_significance)],
    ["normal_model_loglik", float(-opt.fun)],
    ["normal_model_n_params", int(p + k)],
    ["normal_model_AIC", float(2 * (p + k) + 2 * opt.fun)],
    ["normal_model_BIC", float(np.log(len(y)) * (p + k) + 2 * opt.fun)],
    ["normal_model_optimizer_success", str(opt.success)],
    ["normal_model_optimizer_message", str(opt.message)],
    ["n_mean_equation_variables_without_intercept", int(len(X_vars))],
    ["n_variance_equation_variables_without_intercept", int(len(V_vars))],
], columns=["item", "value"])


# =========================================================
# 15) Variable design tables
# =========================================================
variable_design_records = []

for c in X_vars:
    if c == F_COL:
        raw_name = F_COL
        role = "normal_mean_BayesNF_F"
        transformation = "none"
        standardized = False
    elif c in continuous_z_cols:
        raw_name = c.replace("_z", "")
        role = "normal_mean_continuous_X"
        transformation = "x -> z-score"
        standardized = True
    elif c in proportion_logit_z_cols:
        raw_name = c.replace("_logit_z", "")
        role = "normal_mean_proportion_X"
        transformation = "p -> logit(p) -> z-score"
        standardized = True
    else:
        raw_name = c
        role = "normal_mean_X"
        transformation = "unknown"
        standardized = False

    variable_design_records.append({
        "equation": "mean",
        "variable_used_in_model": c,
        "raw_variable": raw_name,
        "role": role,
        "transformation": transformation,
        "standardized": standardized,
    })

for c in V_vars:
    if c in continuous_z_cols:
        raw_name = c.replace("_z", "")
        transformation = "x -> z-score"
        standardized = True
    elif c in proportion_logit_z_cols:
        raw_name = c.replace("_logit_z", "")
        transformation = "p -> logit(p) -> z-score"
        standardized = True
    elif c in ["high_risk", "potential_high_risk"]:
        raw_name = c
        transformation = "raw 0/1 indicator"
        standardized = False
    else:
        raw_name = c
        transformation = "unknown"
        standardized = False

    variable_design_records.append({
        "equation": "log_variance",
        "variable_used_in_model": c,
        "raw_variable": raw_name,
        "role": "log_variance_V",
        "transformation": transformation,
        "standardized": standardized,
    })

variable_design_table = pd.DataFrame(variable_design_records)

reference_table = pd.DataFrame({
    "removed_reference_variable": reference_cols,
    "reason": "reference group to avoid perfect multicollinearity"
})

continuous_transform_table = transform_map[
    transform_map["variable_type"] == "specified_continuous"
].copy()

proportion_transform_table = transform_map[
    transform_map["variable_type"] == "non_specified_continuous_treated_as_proportion"
].copy()


# =========================================================
# 16) Export
# =========================================================
with pd.ExcelWriter(OUT_FILE, engine="openpyxl") as writer:
    coef_table.to_excel(writer, sheet_name="Coefficient_Table", index=False)
    model_info.to_excel(writer, sheet_name="Model_Info", index=False)
    transform_map.to_excel(writer, sheet_name="Transform_Map", index=False)
    continuous_transform_table.to_excel(writer, sheet_name="Continuous_Z_Params", index=False)
    proportion_transform_table.to_excel(writer, sheet_name="Proportion_Logit_Z", index=False)
    variable_design_table.to_excel(writer, sheet_name="Variable_Design", index=False)
    reference_table.to_excel(writer, sheet_name="Reference_Cols", index=False)
    dat.to_excel(writer, sheet_name="Model_Used_Data", index=False)
    normal_dat.to_excel(writer, sheet_name="Normal_Component_Data", index=False)

    coef_table[
        (coef_table["sig"].isin(["***", "**", "*", "."])) |
        (coef_table["parameter"].isin(["alpha_p", "beta_p", "lambda"]))
    ].to_excel(writer, sheet_name="Significant_Only", index=False)


# =========================================================
# 17) Print results
# =========================================================
print("\n==============================")
print("Coefficient table")
print("==============================")
print(coef_table)

print("\n==============================")
print("Model info")
print("==============================")
print(model_info)

print("\n==============================")
print("Mean equation variables")
print("==============================")
print(X_vars)

print("\n==============================")
print("Variance equation variables")
print("==============================")
print(V_vars)

print("\n==============================")
print("Continuous z variables")
print("==============================")
print(continuous_z_cols)

print("\n==============================")
print("Proportion logit-z variables")
print("==============================")
print(proportion_logit_z_cols)

print(f"\nSaved: {OUT_FILE}")

資料筆數： 1297
資料欄位數： 90
轉換前可用資料筆數： 1290
膨脹點筆數： 422
非膨脹點筆數： 868
指定連續型 z-score 變數數量： 13
比例變數 logit-z 變數數量： 20
進入 mixture model 的資料筆數： 1290
進入 mixture model 的膨脹點筆數： 422
進入 mixture model 的非膨脹點筆數： 868
Mean equation X_vars 數量： 34
Variance equation V_vars 數量： 9

Coefficient table
                component    parameter  \
0   inflation probability      alpha_p   
1   inflation probability       beta_p   
2             normal mean       beta_0   
3             normal mean       lambda   
4             normal mean       beta_x   
5             normal mean       beta_x   
6             normal mean       beta_x   
7             normal mean       beta_x   
8             normal mean       beta_x   
9             normal mean       beta_x   
10            normal mean       beta_x   
11            normal mean       beta_x   
12            normal mean       beta_x   
13            normal mean       beta_x   
14            normal mean       beta_x   
15            normal mean       beta_x   
16            n